# Full-gallery candidate retrieval ablation (4 GB GPU)

This experiment reuses the local v4 checkpoint and cached gallery embeddings. It
compares the current global-chunk FAISS gate with a local-frame-mean-per-chunk
index and their union on the complete 300-song gallery. Set `QUERY_SPLIT` to
`val` for iteration or `test` for one final held-out report. Each split writes
to a separate output directory. Query embeddings are cached by checkpoint and
audio fingerprints; never choose production settings from test.


In [ ]:
from pathlib import Path
import os

# Edit before Run All. This notebook does not train or overwrite training artifacts.
PROJECT_ROOT_OVERRIDE = None
CHECKPOINT_OVERRIDE = None  # Optional absolute path to a compatible v4 best checkpoint.
GALLERY_MODE = 'full'  # 'test': split references; 'full': original music database.
GALLERY_MANIFEST = None  # Optional JSON mapping song IDs to audio path(s) for full mode.
EVAL_BATCH_SIZE = 4
GALLERY_SONG_BLOCK = 8
REFERENCE_FRAME_BLOCK = 512
LOCAL_RAM_CACHE_MB = 256
COMPARE_ZERO_SHOT = True
COMPARE_KAGGLE = True
RUN_FAISS_BENCHMARK = True  # Requires faiss-cpu; exact evaluation works without FAISS.
FAISS_CHUNK_K = 50  # Neighbors per query chunk, not unique songs.
MAX_EVAL_QUERIES = None  # None = all queries; subsets are explicitly labeled.
RUN_DTW_BENCHMARK = False  # CPU-intensive; never needed for the exact baseline.
DTW_MAX_QUERIES = 50  # None = all queries; optional benchmark has a separate denominator.
DTW_CANDIDATES = 10
DTW_WEIGHT = 0.40
DTW_MAX_FRAMES = 4000
QUERY_AUDIO_PATH = None  # Set a local humming WAV/MP3 path for the final search demo.

if PROJECT_ROOT_OVERRIDE is None:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / '02_stage2_humming_qbh').is_dir()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Set PROJECT_ROOT_OVERRIDE to your project root.')
else:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).resolve()
STAGE2_ROOT = PROJECT_ROOT / '02_stage2_humming_qbh'
if GALLERY_MODE not in ('test', 'full'):
    raise ValueError("GALLERY_MODE must be 'test' or 'full'.")
if min(EVAL_BATCH_SIZE, GALLERY_SONG_BLOCK, REFERENCE_FRAME_BLOCK, FAISS_CHUNK_K, DTW_CANDIDATES, DTW_MAX_FRAMES) < 1:
    raise ValueError('Batch/candidate/frame limits must be positive.')
if not 0 <= DTW_WEIGHT <= 1:
    raise ValueError('DTW_WEIGHT must lie in [0,1].')
for limit in (MAX_EVAL_QUERIES, DTW_MAX_QUERIES):
    if limit is not None and limit < 1:
        raise ValueError('Query limits must be positive or None.')
OUTPUT_DIR = STAGE2_ROOT / 'runs' / 'local_4gb' / ('retrieval_ablation_' + GALLERY_MODE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for env, folder in [('NUMBA_CACHE_DIR', 'numba'), ('MPLCONFIGDIR', 'matplotlib')]:
    directory = STAGE2_ROOT / 'cache' / folder
    directory.mkdir(parents=True, exist_ok=True)
    os.environ[env] = str(directory)
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
print('Project:', PROJECT_ROOT)
print('Evaluation output:', OUTPUT_DIR)


In [ ]:
import json
import hashlib
import gc
import time
import random
import sys
import logging
import math
from collections import OrderedDict
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Union
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import soundfile as sf
import librosa
import scipy.sparse
import typeguard
if not hasattr(typeguard, 'check_argument_types'):
    raise ImportError('CoverHunter requires typeguard<3 in this notebook kernel.')
try:
    import faiss
except ImportError:
    faiss = None
    if RUN_FAISS_BENCHMARK:
        print('FAISS benchmark skipped: install faiss-cpu in this kernel to enable it.')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = device.type == 'cuda'
if AMP_ENABLED:
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
print('PyTorch:', torch.__version__, '| Device:', device)
if AMP_ENABLED:
    print(torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
DATASET_SPLIT_DIR = STAGE2_ROOT / 'datasets' / 'dataset_split'
TRAIN_DIR, VAL_DIR, TEST_DIR = [DATASET_SPLIT_DIR / name for name in ('train', 'val', 'test')]
MUSIC_DB_DIR = STAGE2_ROOT / 'datasets' / '2_music_database'
for directory in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    if not directory.is_dir():
        raise FileNotFoundError(directory)
stage1_candidates = [STAGE2_ROOT / 'checkpoints' / 'coverhunter_shs100k_pretrained.pth',
                     PROJECT_ROOT / '01_stage1_cover_song_csi' / 'checkpoints' / 'coverhunter_shs100k_pretrained.pth']
STAGE1_CKPT = next((str(p) for p in stage1_candidates if p.is_file()), None)
finetuned_ckpt_path = Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else (
    STAGE2_ROOT / 'runs' / 'local_4gb' / 'checkpoints' / 'conformer_bytehum_full_d128_timegrid_lat_v4_local4gb_best.pth')
if not finetuned_ckpt_path.is_file():
    raise FileNotFoundError(finetuned_ckpt_path)
if COMPARE_ZERO_SHOT and STAGE1_CKPT is None:
    raise FileNotFoundError('Missing Stage-1 checkpoint for zero-shot comparison.')
print('Fine-tuned checkpoint:', finetuned_ckpt_path)


In [ ]:
# CoverHunter backbone is embedded to keep this notebook self-contained.
import logging
import math
from typing import Union
from typeguard import check_argument_types

class BasicModel(torch.nn.Module):
  """Minimal CoverHunter base class required by the embedded inference model."""

  def model_size(self) -> int:
    return sum(parameter.numel() for parameter in self.parameters())



class Linear(torch.nn.Module):
  def __init__(self, in_dim, out_dim, bias=True, w_init_gain='linear'):
    super(Linear, self).__init__()
    self.linear_layer = torch.nn.Linear(in_dim, out_dim, bias=bias)

    torch.nn.init.xavier_uniform_(
      self.linear_layer.weight,
      gain=torch.nn.init.calculate_gain(w_init_gain))
    return

  def forward(self, x):
    return self.linear_layer(x)


class Conv1d(torch.nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=1, stride=1,
               padding=None, dilation=1, bias=True, w_init_gain='linear'):
    super(Conv1d, self).__init__()
    if padding is None:
      assert (kernel_size % 2 == 1)
      padding = int(dilation * (kernel_size - 1) / 2)

    self.conv = torch.nn.Conv1d(in_channels, out_channels,
                                kernel_size=kernel_size, stride=stride,
                                padding=padding, dilation=dilation,
                                bias=bias)

    torch.nn.init.xavier_uniform_(
      self.conv.weight, gain=torch.nn.init.calculate_gain(w_init_gain))
    return

  def forward(self, signal):
    conv_signal = self.conv(signal)
    return conv_signal


class CenterLoss(nn.Module):
  """Center loss.

  Reference:
    A Discriminative Feature Learning Approach for Deep Face Recognition.

  Args:
      num_classes (int): number of classes.
      feat_dim (int): feature dimension.
  """

  def __init__(self, num_classes=10, feat_dim=2, use_gpu=True):
    super(CenterLoss, self).__init__()
    self.num_classes = num_classes
    self.feat_dim = feat_dim
    self.use_gpu = use_gpu
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    if self.use_gpu:
      self.centers = nn.Parameter(
        torch.randn(self.num_classes, self.feat_dim).cuda())
      self.centers = nn.Parameter(
        torch.randn(self.num_classes, self.feat_dim).to("cuda"))
    else:
      self.centers = nn.Parameter(torch.randn(self.num_classes, self.feat_dim))
    logging.info("CenterLoss: num_classes({}), feat_dim({})".format(
      num_classes, feat_dim))
    return

  def forward(self, x, labels):
    """
    Args:
        x: feature matrix with shape (batch_size, feat_dim).
        labels: ground truth labels with shape (batch_size).
    """
    batch_size = x.size(0)
    distmat = torch.pow(x, 2).sum(dim=1, keepdim=True). \
                expand(batch_size, self.num_classes) + \
              torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(
                self.num_classes, batch_size).t()
    distmat.addmm_(1, -2, x, self.centers.t())

    classes = torch.arange(self.num_classes).long()
    if self.use_gpu:
      classes = classes.cuda()
    labels = labels.unsqueeze(1).expand(batch_size, self.num_classes)
    mask = labels.eq(classes.expand(batch_size, self.num_classes))

    dist = distmat * mask.float()
    loss = dist.clamp(min=1e-12, max=1e+12).sum() / batch_size

    return loss


class FocalLoss(nn.Module):
  """ Focal Loss implement for https://arxiv.org/abs/1708.02002 """

  def __init__(self, gamma: float = 2., alpha: Optional[List] = None, num_cls: int = -1,
               reduction: str = 'mean'):

    super(FocalLoss, self).__init__()
    assert check_argument_types()
    if reduction not in ['mean', 'sum']:
      raise NotImplementedError(
        'Reduction {} not implemented.'.format(reduction))
    self._reduction = reduction
    self._alpha = alpha
    self._gamma = gamma
    if alpha is not None:
      assert len(alpha) <= num_cls, "{} != {}".format(len(alpha), num_cls)
      self._alpha = torch.tensor(self._alpha)
    self._eps = np.finfo(float).eps
    logging.info("Init Focal loss with gamma:{}".format(gamma))
    return

  def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """ compute focal loss for pred and label

    Args:
      y_pred: [batch_size, num_cls]
      y_true: [batch_size]

    Returns:
      loss
    """
    b = y_pred.size(0)
    y_pred_softmax = torch.nn.Softmax(dim=1)(y_pred) + self._eps
    ce = -torch.log(y_pred_softmax)
    ce = ce.gather(1, y_true.view(-1, 1))

    y_pred_softmax = y_pred_softmax.gather(1, y_true.view(-1, 1))
    weight = torch.pow(torch.sub(1., y_pred_softmax), self._gamma)

    if self._alpha is not None:
      self._alpha = self._alpha.to(y_pred.device)
      alpha = self._alpha.gather(0, y_true.view(-1))
      alpha = alpha.unsqueeze(1)
      alpha = alpha / torch.sum(alpha) * b
      weight = torch.mul(alpha, weight)
    fl_loss = torch.mul(weight, ce).squeeze(1)
    return self._reduce(fl_loss)

  def forward_with_onehot(self, y_pred: torch.Tensor,
                          y_true: torch.Tensor) -> torch.Tensor:
    """ Another implement for "forward" with onehot label matrix.

    It is not good because when ce_embed size is large, more memory will used.

    Args:
      y_pred: [batch_size, num_cls]
      y_true: [batch_size]

    Returns:
      loss

    """
    y_pred = torch.nn.Softmax(dim=1)(y_pred)
    y_true = torch.nn.functional.one_hot(y_true, self.num_cls)

    eps = np.finfo(float).eps
    y_pred = y_pred + eps
    ce = torch.mul(y_true, -torch.log(y_pred))
    weight = torch.mul(y_true, torch.pow(torch.sub(1., y_pred), self._gamma))
    fl_loss = torch.mul(self.alpha, torch.mul(weight, ce))

    fl_loss, _ = torch.max(fl_loss, dim=1)
    return self._reduce(fl_loss)

  def _reduce(self, x):
    if self._reduction == 'mean':
      return torch.mean(x)
    else:
      return torch.sum(x)


class HardTripletLoss(nn.Module):
  """Hard/Hardest Triplet Loss
    (pytorch implementation of https://omoindrot.github.io/triplet-loss)
    For each anchor, we get the hardest positive and hardest negative to form a triplet.
    """

  def __init__(self, margin=0.1):
    """ Args:
      margin: margin for triplet loss
    """
    super(HardTripletLoss, self).__init__()
    self._margin = margin
    return

  def forward(self, embeddings, labels):
    """
    Args:
        labels: labels of the batch, of size (batch_size,)
        embeddings: tensor of shape (batch_size, embed_dim)

    Returns:
        triplet_loss: scalar tensor containing the triplet loss
    """
    pairwise_dist = self._pairwise_distance(embeddings, squared=False)

    mask_anchor_positive = self._get_anchor_positive_triplet_mask(
      labels).float()
    valid_positive_dist = pairwise_dist * mask_anchor_positive
    hardest_positive_dist, _ = torch.max(valid_positive_dist, dim=1,
                                         keepdim=True)

    # Get the hardest negative pairs
    mask_negative = self._get_anchor_negative_triplet_mask(labels).float()
    max_negative_dist, _ = torch.max(pairwise_dist, dim=1, keepdim=True)
    negative_dist = pairwise_dist + max_negative_dist * (1.0 - mask_negative)
    hardest_negative_dist, _ = torch.min(negative_dist, dim=1, keepdim=True)

    # Combine biggest d(a, p) and smallest d(a, n) into final triplet loss
    triplet_loss = F.relu(
      hardest_positive_dist - hardest_negative_dist + self._margin)
    triplet_loss = torch.mean(triplet_loss)
    return triplet_loss

  @staticmethod
  def _pairwise_distance(x, squared=False, eps=1e-16):
    # Compute the 2D matrix of distances between all the embeddings.

    cor_mat = torch.matmul(x, x.t())
    norm_mat = cor_mat.diag()
    distances = norm_mat.unsqueeze(1) - 2 * cor_mat + norm_mat.unsqueeze(0)
    distances = F.relu(distances)

    if not squared:
      mask = torch.eq(distances, 0.0).float()
      distances = distances + mask * eps
      distances = torch.sqrt(distances)
      distances = distances * (1.0 - mask)
    return distances

  @staticmethod
  def _get_anchor_positive_triplet_mask(labels):
    """Return a 2D mask where mask[a, p] is True, if a and p are distinct and
       have same label.

    """
    # device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    device = labels.device
    indices_not_equal = torch.eye(labels.shape[0]).to(device).byte() ^ 1
    labels_equal = torch.unsqueeze(labels, 0) == torch.unsqueeze(labels, 1)
    mask = indices_not_equal * labels_equal
    return mask

  @staticmethod
  def _get_anchor_negative_triplet_mask(labels):
    """Return a 2D mask where mask[a, n] is True iff a and n have distinct labels.
    """
    labels_equal = torch.unsqueeze(labels, 0) == torch.unsqueeze(labels, 1)
    mask = labels_equal ^ 1
    return mask


def subsequent_chunk_mask(
    size: int,
    chunk_size: int,
    num_left_chunks: int = -1,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
  """Create mask for subsequent steps (size, size) with chunk size,
       this is for streaming encoder

    Args:
        size (int): size of mask
        chunk_size (int): size of chunk
        num_left_chunks (int): number of left chunks
            <0: use full chunk
            >=0: use num_left_chunks
        device (torch.device): "cpu" or "cuda" or torch.Tensor.device

    Returns:
        torch.Tensor: mask

    Examples:
        >>> subsequent_chunk_mask(4, 2)
        [[1, 1, 0, 0],
         [1, 1, 0, 0],
         [1, 1, 1, 1],
         [1, 1, 1, 1]]
    """
  ret = torch.zeros(size, size, device=device, dtype=torch.bool)
  for i in range(size):
    if num_left_chunks < 0:
      start = 0
    else:
      start = max((i // chunk_size - num_left_chunks) * chunk_size, 0)
    ending = min((i // chunk_size + 1) * chunk_size, size)
    ret[i, start:ending] = True
  return ret


def add_optional_chunk_mask(xs: torch.Tensor, masks: torch.Tensor,
                            use_dynamic_chunk: bool,
                            use_dynamic_left_chunk: bool,
                            decoding_chunk_size: int, static_chunk_size: int,
                            num_decoding_left_chunks: int):
  """ Apply optional mask for encoder.

    Args:
        masks:
        xs (torch.Tensor): padded input, (B, L, D), L for max length
        mask (torch.Tensor): mask for xs, (B, 1, L)
        use_dynamic_chunk (bool): whether to use dynamic chunk or not
        use_dynamic_left_chunk (bool): whether to use dynamic left chunk for
            training.
        decoding_chunk_size (int): decoding chunk size for dynamic chunk, it's
            0: default for training, use random dynamic chunk.
            <0: for decoding, use full chunk.
            >0: for decoding, use fixed chunk size as set.
        static_chunk_size (int): chunk size for static chunk training/decoding
            if it's greater than 0, if use_dynamic_chunk is true,
            this parameter will be ignored
        num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
            >=0: use num_decoding_left_chunks
            <0: use all left chunks

    Returns:
        torch.Tensor: chunk mask of the input xs.
    """
  # Whether to use chunk mask or not
  if use_dynamic_chunk:
    max_len = xs.size(1)
    if decoding_chunk_size < 0:
      chunk_size = max_len
      num_left_chunks = -1
    elif decoding_chunk_size > 0:
      chunk_size = decoding_chunk_size
      num_left_chunks = num_decoding_left_chunks
    else:
      # chunk size is either [1, 25] or full context(max_len).
      # Since we use 4 times subsampling and allow up to 1s(100 frames)
      # delay, the maximum frame is 100 / 4 = 25.
      chunk_size = torch.randint(1, max_len, (1,)).item()
      num_left_chunks = -1
      if chunk_size > max_len // 2:
        chunk_size = max_len
      else:
        chunk_size = chunk_size % 25 + 1
        if use_dynamic_left_chunk:
          max_left_chunks = (max_len - 1) // chunk_size
          num_left_chunks = torch.randint(0, max_left_chunks,
                                          (1,)).item()
    chunk_masks = subsequent_chunk_mask(xs.size(1), chunk_size,
                                        num_left_chunks,
                                        xs.device)  # (L, L)
    chunk_masks = chunk_masks.unsqueeze(0)  # (1, L, L)
    chunk_masks = masks & chunk_masks  # (B, L, L)
  elif static_chunk_size > 0:
    num_left_chunks = num_decoding_left_chunks
    chunk_masks = subsequent_chunk_mask(xs.size(1), static_chunk_size,
                                        num_left_chunks,
                                        xs.device)  # (L, L)
    chunk_masks = chunk_masks.unsqueeze(0)  # (1, L, L)
    chunk_masks = masks & chunk_masks  # (B, L, L)
  else:
    chunk_masks = masks
  return chunk_masks


def make_pad_mask(lengths: torch.Tensor, max_len: int = 0) -> torch.Tensor:
  """Make mask tensor containing indices of padded part.

    See description of make_non_pad_mask.

    Args:
        lengths (torch.Tensor): Batch of lengths (B,).
        max_len:

    Returns:
        torch.Tensor: Mask tensor containing indices of padded part.

    Examples:
        >>> make_pad_mask(torch.tensor([5, 3, 2]))
        masks = [[0, 0, 0, 0 ,0],
                 [0, 0, 0, 1, 1],
                 [0, 0, 1, 1, 1]]
    """
  batch_size = lengths.size(0)
  max_len = max_len if max_len > 0 else lengths.max().item()
  seq_range = torch.arange(0,
                           max_len,
                           dtype=torch.int64,
                           device=lengths.device)
  seq_range_expand = seq_range.unsqueeze(0).expand(batch_size, max_len)
  seq_length_expand = lengths.unsqueeze(-1)
  mask = seq_range_expand >= seq_length_expand
  return mask


class Swish(torch.nn.Module):
  """Construct an Swish object."""

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """Return Swish activation function."""
    return x * torch.sigmoid(x)


def get_activation(act):
  """Return activation function."""
  activation_funcs = {
    "hardtanh": torch.nn.Hardtanh,
    "tanh": torch.nn.Tanh,
    "relu": torch.nn.ReLU,
    "selu": torch.nn.SELU,
    "swish": getattr(torch.nn, "SiLU", Swish),
    "gelu": torch.nn.GELU
  }
  return activation_funcs[act]()


class PositionalEncoding(torch.nn.Module):
  """Positional encoding.

    :param int d_model: embedding dim
    :param float dropout_rate: dropout rate
    :param int max_len: maximum input length

    PE(pos, 2i)   = sin(pos/(10000^(2i/dmodel)))
    PE(pos, 2i+1) = cos(pos/(10000^(2i/dmodel)))
    """

  def __init__(self,
               d_model: int,
               dropout_rate: float,
               max_len: int = 5000,
               reverse: bool = False):
    """Construct an PositionalEncoding object."""
    super().__init__()
    self.d_model = d_model
    self.xscale = math.sqrt(self.d_model)
    self.dropout = torch.nn.Dropout(p=dropout_rate)
    self.max_len = max_len

    self.pe = torch.zeros(self.max_len, self.d_model)
    position = torch.arange(0, self.max_len,
                            dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
      torch.arange(0, self.d_model, 2, dtype=torch.float32) *
      -(math.log(10000.0) / self.d_model))
    self.pe[:, 0::2] = torch.sin(position * div_term)
    self.pe[:, 1::2] = torch.cos(position * div_term)
    self.pe = self.pe.unsqueeze(0)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """Add positional encoding.

        Args:
            x (torch.Tensor): Input. Its shape is (batch, time, ...)
            offset (int, torch.tensor): position offset

        Returns:
            torch.Tensor: Encoded tensor. Its shape is (batch, time, ...)
            torch.Tensor: for compatibility to RelPositionalEncoding
        """

    self.pe = self.pe.to(x.device)
    pos_emb = self.position_encoding(offset, x.size(1), False)
    x = x * self.xscale + pos_emb
    return self.dropout(x), self.dropout(pos_emb)

  def position_encoding(self, offset: Union[int, torch.Tensor], size: int,
                        apply_dropout: bool = True) -> torch.Tensor:
    """ For getting encoding in a streaming fashion

        Attention!!!!!
        we apply dropout only once at the whole utterance level in a none
        streaming way, but will call this function several times with
        increasing input size in a streaming scenario, so the dropout will
        be applied several times.

        Args:
            apply_dropout:
            offset (int or torch.tensor): start offset
            size (int): required size of position encoding

        Returns:
            torch.Tensor: Corresponding encoding
        """
    # How to subscript a Union type:
    #   https://github.com/pytorch/pytorch/issues/69434
    if isinstance(offset, int):
      assert offset + size < self.max_len
      pos_emb = self.pe[:, offset:offset + size]
    elif isinstance(offset, torch.Tensor) and offset.dim() == 0:  # scalar
      assert offset + size < self.max_len
      pos_emb = self.pe[:, offset:offset + size]
    else:  # for batched streaming decoding on GPU
      assert torch.max(offset) + size < self.max_len
      index = offset.unsqueeze(1) + torch.arange(0, size).to(offset.device)  # B X T
      flag = index > 0
      # remove negative offset
      index = index * flag
      pos_emb = F.embedding(index, self.pe[0])  # B X T X d_model

    if apply_dropout:
      pos_emb = self.dropout(pos_emb)
    return pos_emb


class RelPositionalEncoding(PositionalEncoding):
  """Relative positional encoding module.
    See : Appendix B in https://arxiv.org/abs/1901.02860
    Args:
        d_model (int): Embedding dimension.
        dropout_rate (float): Dropout rate.
        max_len (int): Maximum input length.
    """

  def __init__(self, d_model: int, dropout_rate: float, max_len: int = 5000):
    """Initialize class."""
    super().__init__(d_model, dropout_rate, max_len, reverse=True)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute positional encoding.
        Args:
            x (torch.Tensor): Input tensor (batch, time, `*`).
        Returns:
            torch.Tensor: Encoded tensor (batch, time, `*`).
            torch.Tensor: Positional embedding tensor (1, time, `*`).
        """
    self.pe = self.pe.to(x.device)
    x = x * self.xscale
    pos_emb = self.position_encoding(offset, x.size(1), False)
    return self.dropout(x), self.dropout(pos_emb)


class NoPositionalEncoding(torch.nn.Module):
  """ No position encoding
    """

  def __init__(self, d_model: int, dropout_rate: float):
    super().__init__()
    self.d_model = d_model
    self.dropout = torch.nn.Dropout(p=dropout_rate)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """ Just return zero vector for interface compatibility
        """
    pos_emb = torch.zeros(1, x.size(1), self.d_model).to(x.device)
    return self.dropout(x), pos_emb

  def position_encoding(
      self, offset: Union[int, torch.Tensor], size: int) -> torch.Tensor:
    return torch.zeros(1, size, self.d_model)


class BaseSubsampling(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.right_context = 0
    self.subsampling_rate = 1

  def position_encoding(self, offset: Union[int, torch.Tensor],
                        size: int) -> torch.Tensor:
    return self.pos_enc.position_encoding(offset, size)


class LinearNoSubsampling(BaseSubsampling):
  """Linear transform the input without subsampling

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an linear object."""
    super().__init__()
    self.out = torch.nn.Sequential(
      torch.nn.Linear(idim, odim),
      torch.nn.LayerNorm(odim, eps=1e-5),
      torch.nn.Dropout(dropout_rate),
    )
    self.pos_enc = pos_enc_class
    self.right_context = 0
    self.subsampling_rate = 1

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Input x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: linear input tensor (#batch, time', odim),
                where time' = time .
            torch.Tensor: linear input mask (#batch, 1, time'),
                where time' = time .

        """
    x = self.out(x)
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask


class Conv2dSubsampling4(BaseSubsampling):
  """Convolutional 2D subsampling (to 1/4 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an Conv2dSubsampling4 object."""
    super().__init__()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv2d(1, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
    )
    self.out = torch.nn.Sequential(
      torch.nn.Linear(odim * (((idim - 1) // 2 - 1) // 2), odim))
    self.pos_enc = pos_enc_class
    # The right context for every conv layer is computed by:
    # (kernel_size - 1) * frame_rate_of_this_layer
    self.subsampling_rate = 4
    # 6 = (3 - 1) * 1 + (3 - 1) * 2
    self.right_context = 6

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 4.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 4.
            torch.Tensor: positional encoding

        """
    x = x.unsqueeze(1)  # (b, c=1, t, f)
    x = self.conv(x)
    b, c, t, f = x.size()
    x = self.out(x.transpose(1, 2).contiguous().view(b, t, c * f))
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask[:, :, :-2:2][:, :, :-2:2]


class Conv2dSubsampling8(BaseSubsampling):
  """Convolutional 2D subsampling (to 1/8 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an Conv2dSubsampling8 object."""
    super().__init__()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv2d(1, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
    )
    self.linear = torch.nn.Linear(
      odim * ((((idim - 1) // 2 - 1) // 2 - 1) // 2), odim)
    self.pos_enc = pos_enc_class
    self.subsampling_rate = 8
    # 14 = (3 - 1) * 1 + (3 - 1) * 2 + (3 - 1) * 4
    self.right_context = 14

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 8.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 8.
            torch.Tensor: positional encoding
        """
    x = x.unsqueeze(1)  # (b, c, t, f)
    x = self.conv(x)
    b, c, t, f = x.size()
    x = self.linear(x.transpose(1, 2).contiguous().view(b, t, c * f))
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask[:, :, :-2:2][:, :, :-2:2][:, :, :-2:2]


class MultiHeadedAttention(nn.Module):
  """Multi-Head Attention layer.

    Args:
        n_head (int): The number of heads.
        n_feat (int): The number of features.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, n_head: int, n_feat: int, dropout_rate: float):
    """Construct an MultiHeadedAttention object."""
    super().__init__()
    assert n_feat % n_head == 0
    # We assume d_v always equals d_k
    self.d_k = n_feat // n_head
    self.h = n_head
    self.linear_q = nn.Linear(n_feat, n_feat)
    self.linear_k = nn.Linear(n_feat, n_feat)
    self.linear_v = nn.Linear(n_feat, n_feat)
    self.linear_out = nn.Linear(n_feat, n_feat)
    self.dropout = nn.Dropout(p=dropout_rate)

  def forward_qkv(
      self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Transform query, key and value.

        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).

        Returns:
            torch.Tensor: Transformed query tensor, size
                (#batch, n_head, time1, d_k).
            torch.Tensor: Transformed key tensor, size
                (#batch, n_head, time2, d_k).
            torch.Tensor: Transformed value tensor, size
                (#batch, n_head, time2, d_k).

        """
    n_batch = query.size(0)
    q = self.linear_q(query).view(n_batch, -1, self.h, self.d_k)
    k = self.linear_k(key).view(n_batch, -1, self.h, self.d_k)
    v = self.linear_v(value).view(n_batch, -1, self.h, self.d_k)
    q = q.transpose(1, 2)  # (batch, head, time1, d_k)
    k = k.transpose(1, 2)  # (batch, head, time2, d_k)
    v = v.transpose(1, 2)  # (batch, head, time2, d_k)

    return q, k, v

  def forward_attention(
      self, value: torch.Tensor, scores: torch.Tensor,
      mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool)
  ) -> torch.Tensor:
    """Compute attention context vector.

        Args:
            value (torch.Tensor): Transformed value, size
                (#batch, n_head, time2, d_k).
            scores (torch.Tensor): Attention score, size
                (#batch, n_head, time1, time2).
            mask (torch.Tensor): Mask, size (#batch, 1, time2) or
                (#batch, time1, time2), (0, 0, 0) means fake mask.

        Returns:
            torch.Tensor: Transformed value (#batch, time1, d_model)
                weighted by the attention score (#batch, time1, time2).

        """
    n_batch = value.size(0)
    # NOTE(xcsong): When will `if mask.size(2) > 0` be True?
    #   1. onnx(16/4) [WHY? Because we feed real cache & real mask for the
    #           1st chunk to ease the onnx export.]
    #   2. pytorch training
    if mask.size(2) > 0:  # time2 > 0
      mask = mask.unsqueeze(1).eq(0)  # (batch, 1, *, time2)
      # For last chunk, time2 might be larger than scores.size(-1)
      mask = mask[:, :, :, :scores.size(-1)]  # (batch, 1, *, time2)
      scores = scores.masked_fill(mask, -float('inf'))
      attn = torch.softmax(scores, dim=-1).masked_fill(
        mask, 0.0)  # (batch, head, time1, time2)
    # NOTE(xcsong): When will `if mask.size(2) > 0` be False?
    #   1. onnx(16/-1, -1/-1, 16/0)
    #   2. jit (16/-1, -1/-1, 16/0, 16/4)
    else:
      attn = torch.softmax(scores, dim=-1)  # (batch, head, time1, time2)

    p_attn = self.dropout(attn)
    x = torch.matmul(p_attn, value)  # (batch, head, time1, d_k)
    x = (x.transpose(1, 2).contiguous().view(n_batch, -1,
                                             self.h * self.d_k)
         )  # (batch, time1, d_model)

    return self.linear_out(x)  # (batch, time1, d_model)

  def forward(self, query: torch.Tensor, key: torch.Tensor,
              value: torch.Tensor,
              mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
              pos_emb: torch.Tensor = torch.empty(0),
              cache: torch.Tensor = torch.zeros((0, 0, 0, 0))
              ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot product attention.

        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).
            mask (torch.Tensor): Mask tensor (#batch, 1, time2) or
                (#batch, time1, time2).
                1.When applying cross attention between decoder and encoder,
                the batch padding mask for input is in (#batch, 1, T) shape.
                2.When applying self attention of encoder,
                the mask is in (#batch, T, T)  shape.
                3.When applying self attention of decoder,
                the mask is in (#batch, L, L)  shape.
                4.If the different position in decoder see different block
                of the encoder, such as Mocha, the passed in mask could be
                in (#batch, L, T) shape. But there is no such case in current
                Wenet.
            cache (torch.Tensor): Cache tensor (1, head, cache_t, d_k * 2),
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`


        Returns:
            torch.Tensor: Output tensor (#batch, time1, d_model).
            torch.Tensor: Cache tensor (1, head, cache_t + time1, d_k * 2)
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`

        """
    q, k, v = self.forward_qkv(query, key, value)

    # NOTE(xcsong):
    #   when export onnx model, for 1st chunk, we feed
    #       cache(1, head, 0, d_k * 2) (16/-1, -1/-1, 16/0 mode)
    #       or cache(1, head, real_cache_t, d_k * 2) (16/4 mode).
    #       In all modes, `if cache.size(0) > 0` will alwayse be `True`
    #       and we will always do splitting and
    #       concatnation(this will simplify onnx export). Note that
    #       it's OK to concat & split zero-shaped tensors(see code below).
    #   when export jit  model, for 1st chunk, we always feed
    #       cache(0, 0, 0, 0) since jit supports dynamic if-branch.
    # >>> a = torch.ones((1, 2, 0, 4))
    # >>> b = torch.ones((1, 2, 3, 4))
    # >>> c = torch.cat((a, b), dim=2)
    # >>> torch.equal(b, c)        # True
    # >>> d = torch.split(a, 2, dim=-1)
    # >>> torch.equal(d[0], d[1])  # True
    if cache.size(0) > 0:
      key_cache, value_cache = torch.split(
        cache, cache.size(-1) // 2, dim=-1)
      k = torch.cat([key_cache, k], dim=2)
      v = torch.cat([value_cache, v], dim=2)
    # NOTE(xcsong): We do cache slicing in encoder.forward_chunk, since it's
    #   non-trivial to calculate `next_cache_start` here.
    new_cache = torch.cat((k, v), dim=-1)

    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
    return self.forward_attention(v, scores, mask), new_cache


class RelPositionMultiHeadedAttention(MultiHeadedAttention):
  """Multi-Head Attention layer with relative position encoding.
    Paper: https://arxiv.org/abs/1901.02860
    Args:
        n_head (int): The number of heads.
        n_feat (int): The number of features.
        dropout_rate (float): Dropout rate.
    """

  def __init__(self, n_head, n_feat, dropout_rate):
    """Construct an RelPositionMultiHeadedAttention object."""
    super().__init__(n_head, n_feat, dropout_rate)
    # linear transformation for positional encoding
    self.linear_pos = nn.Linear(n_feat, n_feat, bias=False)
    # these two learnable bias are used in matrix c and matrix d
    # as described in https://arxiv.org/abs/1901.02860 Section 3.3
    self.pos_bias_u = nn.Parameter(torch.Tensor(self.h, self.d_k))
    self.pos_bias_v = nn.Parameter(torch.Tensor(self.h, self.d_k))
    torch.nn.init.xavier_uniform_(self.pos_bias_u)
    torch.nn.init.xavier_uniform_(self.pos_bias_v)

  def rel_shift(self, x, zero_triu: bool = False):
    """Compute relative positinal encoding.
        Args:
            x (torch.Tensor): Input tensor (batch, time, size).
            zero_triu (bool): If true, return the lower triangular part of
                the matrix.
        Returns:
            torch.Tensor: Output tensor.
        """

    zero_pad = torch.zeros((x.size()[0], x.size()[1], x.size()[2], 1),
                           device=x.device,
                           dtype=x.dtype)
    x_padded = torch.cat([zero_pad, x], dim=-1)

    x_padded = x_padded.view(x.size()[0],
                             x.size()[1],
                             x.size(3) + 1, x.size(2))
    x = x_padded[:, :, 1:].view_as(x)

    if zero_triu:
      ones = torch.ones((x.size(2), x.size(3)))
      x = x * torch.tril(ones, x.size(3) - x.size(2))[None, None, :, :]

    return x

  def forward(self, query: torch.Tensor,
              key: torch.Tensor, value: torch.Tensor,
              mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
              pos_emb: torch.Tensor = torch.empty(0),
              cache: torch.Tensor = torch.zeros((0, 0, 0, 0))
              ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute 'Scaled Dot Product Attention' with rel. positional encoding.
        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).
            mask (torch.Tensor): Mask tensor (#batch, 1, time2) or
                (#batch, time1, time2), (0, 0, 0) means fake mask.
            pos_emb (torch.Tensor): Positional embedding tensor
                (#batch, time2, size).
            cache (torch.Tensor): Cache tensor (1, head, cache_t, d_k * 2),
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`
        Returns:
            torch.Tensor: Output tensor (#batch, time1, d_model).
            torch.Tensor: Cache tensor (1, head, cache_t + time1, d_k * 2)
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`
        """
    q, k, v = self.forward_qkv(query, key, value)
    q = q.transpose(1, 2)  # (batch, time1, head, d_k)

    # NOTE(xcsong):
    #   when export onnx model, for 1st chunk, we feed
    #       cache(1, head, 0, d_k * 2) (16/-1, -1/-1, 16/0 mode)
    #       or cache(1, head, real_cache_t, d_k * 2) (16/4 mode).
    #       In all modes, `if cache.size(0) > 0` will alwayse be `True`
    #       and we will always do splitting and
    #       concatnation(this will simplify onnx export). Note that
    #       it's OK to concat & split zero-shaped tensors(see code below).
    #   when export jit  model, for 1st chunk, we always feed
    #       cache(0, 0, 0, 0) since jit supports dynamic if-branch.
    # >>> a = torch.ones((1, 2, 0, 4))
    # >>> b = torch.ones((1, 2, 3, 4))
    # >>> c = torch.cat((a, b), dim=2)
    # >>> torch.equal(b, c)        # True
    # >>> d = torch.split(a, 2, dim=-1)
    # >>> torch.equal(d[0], d[1])  # True
    if cache.size(0) > 0:
      key_cache, value_cache = torch.split(
        cache, cache.size(-1) // 2, dim=-1)
      k = torch.cat([key_cache, k], dim=2)
      v = torch.cat([value_cache, v], dim=2)
    # NOTE(xcsong): We do cache slicing in encoder.forward_chunk, since it's
    #   non-trivial to calculate `next_cache_start` here.
    new_cache = torch.cat((k, v), dim=-1)

    n_batch_pos = pos_emb.size(0)
    p = self.linear_pos(pos_emb).view(n_batch_pos, -1, self.h, self.d_k)
    p = p.transpose(1, 2)  # (batch, head, time1, d_k)

    # (batch, head, time1, d_k)
    q_with_bias_u = (q + self.pos_bias_u).transpose(1, 2)
    # (batch, head, time1, d_k)
    q_with_bias_v = (q + self.pos_bias_v).transpose(1, 2)

    # compute attention score
    # first compute matrix a and matrix c
    # as described in https://arxiv.org/abs/1901.02860 Section 3.3
    # (batch, head, time1, time2)
    matrix_ac = torch.matmul(q_with_bias_u, k.transpose(-2, -1))

    # compute matrix b and matrix d
    # (batch, head, time1, time2)
    matrix_bd = torch.matmul(q_with_bias_v, p.transpose(-2, -1))
    # Remove rel_shift since it is useless in speech recognition,
    # and it requires special attention for streaming.
    # matrix_bd = self.rel_shift(matrix_bd)

    scores = (matrix_ac + matrix_bd) / math.sqrt(
      self.d_k)  # (batch, head, time1, time2)

    return self.forward_attention(v, scores, mask), new_cache


class ConvolutionModule(nn.Module):
  """ConvolutionModule in Conformer model."""

  def __init__(self,
               channels: int,
               kernel_size: int = 15,
               activation: nn.Module = nn.ReLU(),
               norm: str = "batch_norm",
               causal: bool = False,
               bias: bool = True):
    """Construct an ConvolutionModule object.
        Args:
            channels (int): The number of channels of conv layers.
            kernel_size (int): Kernel size of conv layers.
            causal (int): Whether use causal convolution or not
        """
    assert check_argument_types()
    super().__init__()

    self.pointwise_conv1 = nn.Conv1d(
      channels,
      2 * channels,
      kernel_size=1,
      stride=1,
      padding=0,
      bias=bias,
    )
    # self.lorder is used to distinguish if it's a causal convolution,
    # if self.lorder > 0: it's a causal convolution, the input will be
    #    padded with self.lorder frames on the left in forward.
    # else: it's a symmetrical convolution
    if causal:
      padding = 0
      self.lorder = kernel_size - 1
    else:
      # kernel_size should be an odd number for none causal convolution
      assert (kernel_size - 1) % 2 == 0
      padding = (kernel_size - 1) // 2
      self.lorder = 0
    self.depthwise_conv = nn.Conv1d(
      channels,
      channels,
      kernel_size,
      stride=1,
      padding=padding,
      groups=channels,
      bias=bias,
    )

    assert norm in ['batch_norm', 'layer_norm']
    if norm == "batch_norm":
      self.use_layer_norm = False
      self.norm = nn.BatchNorm1d(channels)
    else:
      self.use_layer_norm = True
      self.norm = nn.LayerNorm(channels)

    self.pointwise_conv2 = nn.Conv1d(
      channels,
      channels,
      kernel_size=1,
      stride=1,
      padding=0,
      bias=bias,
    )
    self.activation = activation

  def forward(
      self,
      x: torch.Tensor,
      mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
      cache: torch.Tensor = torch.zeros((0, 0, 0)),
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute convolution module.
        Args:
            x (torch.Tensor): Input tensor (#batch, time, channels).
            mask_pad (torch.Tensor): used for batch padding (#batch, 1, time),
                (0, 0, 0) means fake mask.
            cache (torch.Tensor): left context cache, it is only
                used in causal convolution (#batch, channels, cache_t),
                (0, 0, 0) meas fake cache.
        Returns:
            torch.Tensor: Output tensor (#batch, time, channels).
        """
    # exchange the temporal dimension and the feature dimension
    x = x.transpose(1, 2)  # (#batch, channels, time)

    # mask batch padding
    if mask_pad.size(2) > 0:  # time > 0
      x.masked_fill_(~mask_pad, 0.0)

    if self.lorder > 0:
      if cache.size(2) == 0:  # cache_t == 0
        x = nn.functional.pad(x, (self.lorder, 0), 'constant', 0.0)
      else:
        assert cache.size(0) == x.size(0)  # equal batch
        assert cache.size(1) == x.size(1)  # equal channel
        x = torch.cat((cache, x), dim=2)
      assert (x.size(2) > self.lorder)
      new_cache = x[:, :, -self.lorder:]
    else:
      # It's better we just return None if no cache is required,
      # However, for JIT export, here we just fake one tensor instead of
      # None.
      new_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)

    # GLU mechanism
    x = self.pointwise_conv1(x)  # (batch, 2*channel, dim)
    x = nn.functional.glu(x, dim=1)  # (batch, channel, dim)

    # 1D Depthwise Conv
    x = self.depthwise_conv(x)
    if self.use_layer_norm:
      x = x.transpose(1, 2)
    x = self.activation(self.norm(x))
    if self.use_layer_norm:
      x = x.transpose(1, 2)
    x = self.pointwise_conv2(x)
    # mask batch padding
    if mask_pad.size(2) > 0:  # time > 0
      x.masked_fill_(~mask_pad, 0.0)

    return x.transpose(1, 2), new_cache


class PositionwiseFeedForward(torch.nn.Module):
  """Positionwise feed forward layer.

    FeedForward are appied on each position of the sequence.
    The output dim is same with the input dim.

    Args:
        idim (int): Input dimenstion.
        hidden_units (int): The number of hidden units.
        dropout_rate (float): Dropout rate.
        activation (torch.nn.Module): Activation function
    """

  def __init__(self,
               idim: int,
               hidden_units: int,
               dropout_rate: float,
               activation: torch.nn.Module = torch.nn.ReLU()):
    """Construct a PositionwiseFeedForward object."""
    super(PositionwiseFeedForward, self).__init__()
    self.w_1 = torch.nn.Linear(idim, hidden_units)
    self.activation = activation
    self.dropout = torch.nn.Dropout(dropout_rate)
    self.w_2 = torch.nn.Linear(hidden_units, idim)

  def forward(self, xs: torch.Tensor) -> torch.Tensor:
    """Forward function.

        Args:
            xs: input tensor (B, L, D)
        Returns:
            output tensor, (B, L, D)
        """
    return self.w_2(self.dropout(self.activation(self.w_1(xs))))


class ConformerEncoderLayer(torch.nn.Module):
  """Encoder layer module.
    Args:
        size (int): Input dimension.
        self_attn (torch.nn.Module): Self-attention module instance.
            `MultiHeadedAttention` or `RelPositionMultiHeadedAttention`
            instance can be used as the argument.
        feed_forward (torch.nn.Module): Feed-forward module instance.
            `PositionwiseFeedForward` instance can be used as the argument.
        feed_forward_macaron (torch.nn.Module): Additional feed-forward module
             instance.
            `PositionwiseFeedForward` instance can be used as the argument.
        conv_module (torch.nn.Module): Convolution module instance.
            `ConvlutionModule` instance can be used as the argument.
        dropout_rate (float): Dropout rate.
        normalize_before (bool):
            True: use layer_norm before each sub-block.
            False: use layer_norm after each sub-block.
        concat_after (bool): Whether to concat attention layer's input and
            output.
            True: x -> x + linear(concat(x, att(x)))
            False: x -> x + att(x)
    """

  def __init__(
      self,
      size: int,
      self_attn: torch.nn.Module,
      feed_forward: Optional[nn.Module] = None,
      feed_forward_macaron: Optional[nn.Module] = None,
      conv_module: Optional[nn.Module] = None,
      dropout_rate: float = 0.1,
      normalize_before: bool = True,
      concat_after: bool = False,
  ):
    """Construct an EncoderLayer object."""
    super().__init__()
    self.self_attn = self_attn
    self.feed_forward = feed_forward
    self.feed_forward_macaron = feed_forward_macaron
    self.conv_module = conv_module
    self.norm_ff = nn.LayerNorm(size, eps=1e-5)  # for the FNN module
    self.norm_mha = nn.LayerNorm(size, eps=1e-5)  # for the MHA module
    if feed_forward_macaron is not None:
      self.norm_ff_macaron = nn.LayerNorm(size, eps=1e-5)
      self.ff_scale = 0.5
    else:
      self.ff_scale = 1.0
    if self.conv_module is not None:
      self.norm_conv = nn.LayerNorm(size,
                                    eps=1e-5)  # for the CNN module
      self.norm_final = nn.LayerNorm(
        size, eps=1e-5)  # for the final output of the block
    self.dropout = nn.Dropout(dropout_rate)
    self.size = size
    self.normalize_before = normalize_before
    self.concat_after = concat_after
    if self.concat_after:
      self.concat_linear = nn.Linear(size + size, size)
    else:
      self.concat_linear = nn.Identity()

  def forward(
      self,
      x: torch.Tensor,
      mask: torch.Tensor,
      pos_emb: torch.Tensor,
      mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
      att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
      cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute encoded features.

        Args:
            x (torch.Tensor): (#batch, time, size)
            mask (torch.Tensor): Mask tensor for the input (#batch, time，time),
                (0, 0, 0) means fake mask.
            pos_emb (torch.Tensor): positional encoding, must not be None
                for ConformerEncoderLayer.
            mask_pad (torch.Tensor): batch padding mask used for conv module.
                (#batch, 1，time), (0, 0, 0) means fake mask.
            att_cache (torch.Tensor): Cache tensor of the KEY & VALUE
                (#batch=1, head, cache_t1, d_k * 2), head * d_k == size.
            cnn_cache (torch.Tensor): Convolution cache in conformer layer
                (#batch=1, size, cache_t2)
        Returns:
            torch.Tensor: Output tensor (#batch, time, size).
            torch.Tensor: Mask tensor (#batch, time, time).
            torch.Tensor: att_cache tensor,
                (#batch=1, head, cache_t1 + time, d_k * 2).
            torch.Tensor: cnn_cahce tensor (#batch, size, cache_t2).
        """

    # whether to use macaron style
    if self.feed_forward_macaron is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_ff_macaron(x)
      x = residual + self.ff_scale * self.dropout(
        self.feed_forward_macaron(x))
      if not self.normalize_before:
        x = self.norm_ff_macaron(x)

    # multi-headed self-attention module
    residual = x
    if self.normalize_before:
      x = self.norm_mha(x)

    x_att, new_att_cache = self.self_attn(
      x, x, x, mask, pos_emb, att_cache)
    if self.concat_after:
      x_concat = torch.cat((x, x_att), dim=-1)
      x = residual + self.concat_linear(x_concat)
    else:
      x = residual + self.dropout(x_att)
    if not self.normalize_before:
      x = self.norm_mha(x)

    # convolution module
    # Fake new cnn cache here, and then change it in conv_module
    new_cnn_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)
    if self.conv_module is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_conv(x)
      x, new_cnn_cache = self.conv_module(x, mask_pad, cnn_cache)
      x = residual + self.dropout(x)

      if not self.normalize_before:
        x = self.norm_conv(x)

    # feed forward module
    residual = x
    if self.normalize_before:
      x = self.norm_ff(x)

    x = residual + self.ff_scale * self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x = self.norm_ff(x)

    if self.conv_module is not None:
      x = self.norm_final(x)

    return x, mask, new_att_cache, new_cnn_cache


class ConformerEncoder(torch.nn.Module):
  def __init__(
      self,
      input_size: int,
      output_size: int,
      attention_heads: int = 4,
      linear_units: int = 2048,
      num_blocks: int = 6,
      dropout_rate: float = 0.1,
      positional_dropout_rate: float = 0.1,
      attention_dropout_rate: float = 0.0,
      input_layer: str = "conv2d",
      pos_enc_layer_type: str = "rel_pos",
      normalize_before: bool = True,
      concat_after: bool = False,
      static_chunk_size: int = 0,
      use_dynamic_chunk: bool = False,
      global_cmvn: Optional[torch.nn.Module] = None,
      use_dynamic_left_chunk: bool = False,
      positionwise_conv_kernel_size: int = 1,
      macaron_style: bool = True,
      selfattention_layer_type: str = "rel_selfattn",
      activation_type: str = "swish",
      use_cnn_module: bool = True,
      cnn_module_kernel: int = 15,
      causal: bool = False,
      cnn_module_norm: str = "batch_norm",
  ):
    """
        Args:
            input_size (int): input dim
            output_size (int): dimension of attention
            attention_heads (int): the number of heads of multi head attention
            linear_units (int): the hidden units number of position-wise feed
                forward
            num_blocks (int): the number of decoder blocks
            dropout_rate (float): dropout rate
            attention_dropout_rate (float): dropout rate in attention
            positional_dropout_rate (float): dropout rate after adding
                positional encoding
            input_layer (str): input layer type.
                optional [linear, conv2d, conv2d6, conv2d8]
            pos_enc_layer_type (str): Encoder positional encoding layer type.
                opitonal [abs_pos, scaled_abs_pos, rel_pos, no_pos]
            normalize_before (bool):
                True: use layer_norm before each sub-block of a layer.
                False: use layer_norm after each sub-block of a layer.
            concat_after (bool): whether to concat attention layer's input
                and output.
                True: x -> x + linear(concat(x, att(x)))
                False: x -> x + att(x)
            static_chunk_size (int): chunk size for static chunk training and
                decoding
            use_dynamic_chunk (bool): whether use dynamic chunk size for
                training or not, You can only use fixed chunk(chunk_size > 0)
                or dyanmic chunk size(use_dynamic_chunk = True)
            global_cmvn (Optional[torch.nn.Module]): Optional GlobalCMVN module
            use_dynamic_left_chunk (bool): whether use dynamic left chunk in
                dynamic chunk training
        """
    assert check_argument_types()
    logging.info("Init Conformer layers::")
    super().__init__()
    self._output_size = output_size

    if pos_enc_layer_type == "abs_pos":
      pos_enc_class = PositionalEncoding
    elif pos_enc_layer_type == "rel_pos":
      pos_enc_class = RelPositionalEncoding
    elif pos_enc_layer_type == "no_pos":
      pos_enc_class = NoPositionalEncoding
    else:
      raise ValueError("unknown pos_enc_layer: " + pos_enc_layer_type)
    logging.info("Pos layers - {}".format(pos_enc_layer_type))

    if input_layer == "linear":
      subsampling_class = LinearNoSubsampling
    elif input_layer == "conv2d":
      subsampling_class = Conv2dSubsampling4
    elif input_layer == "conv2d8":
      subsampling_class = Conv2dSubsampling8
    else:
      raise ValueError("unknown input_layer: " + input_layer)
    logging.info("Input layers - {}".format(input_layer))

    self.global_cmvn = global_cmvn
    self.embed = subsampling_class(
      input_size,
      output_size,
      dropout_rate,
      pos_enc_class(output_size, positional_dropout_rate),
    )

    self.normalize_before = normalize_before
    self.after_norm = torch.nn.LayerNorm(output_size, eps=1e-5)
    self.static_chunk_size = static_chunk_size
    self.use_dynamic_chunk = use_dynamic_chunk
    self.use_dynamic_left_chunk = use_dynamic_left_chunk

    activation = get_activation(activation_type)

    # self-attention module definition
    if pos_enc_layer_type != "rel_pos":
      encoder_selfattn_layer = MultiHeadedAttention
    else:
      encoder_selfattn_layer = RelPositionMultiHeadedAttention
    encoder_selfattn_layer_args = (
      attention_heads,
      output_size,
      attention_dropout_rate,
    )
    logging.info("self attn args: {}".format(encoder_selfattn_layer_args))

    # feed-forward module definition
    positionwise_layer = PositionwiseFeedForward
    positionwise_layer_args = (
      output_size,
      linear_units,
      dropout_rate,
      activation,
    )
    logging.info("position-wise args: {}".format(positionwise_layer_args))

    # convolution module definition
    convolution_layer = ConvolutionModule
    convolution_layer_args = (output_size, cnn_module_kernel, activation,
                              cnn_module_norm, causal)

    self.encoders = torch.nn.ModuleList([
      ConformerEncoderLayer(
        output_size,
        encoder_selfattn_layer(*encoder_selfattn_layer_args),
        positionwise_layer(*positionwise_layer_args),
        positionwise_layer(
          *positionwise_layer_args) if macaron_style else None,
        convolution_layer(
          *convolution_layer_args) if use_cnn_module else None,
        dropout_rate,
        normalize_before,
        concat_after,
      ) for _ in range(num_blocks)
    ])
    return

  def output_size(self) -> int:
    return self._output_size

  def forward(
      self,
      xs: torch.Tensor,
      xs_lens: torch.Tensor,
      decoding_chunk_size: int = -1,
      num_decoding_left_chunks: int = -1,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Embed positions in tensor.

        Args:
            xs: padded input tensor (B, T, D)
            xs_lens: input length (B)
            decoding_chunk_size: decoding chunk size for dynamic chunk
                0: default for training, use random dynamic chunk.
                <0: for decoding, use full chunk.
                >0: for decoding, use fixed chunk size as set.
            num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
                >=0: use num_decoding_left_chunks
                <0: use all left chunks
        Returns:
            encoder output tensor xs, and subsampled masks
            xs: padded output tensor (B, T' ~= T/subsample_rate, D)
            masks: torch.Tensor batch padding mask after subsample
                (B, 1, T' ~= T/subsample_rate)
        """
    T = xs.size(1)
    masks = ~make_pad_mask(xs_lens, T).unsqueeze(1)  # (B, 1, T)
    if self.global_cmvn is not None:
      xs = self.global_cmvn(xs)
    xs, pos_emb, masks = self.embed(xs, masks)
    mask_pad = masks  # (B, 1, T/subsample_rate)
    chunk_masks = add_optional_chunk_mask(xs, masks,
                                          self.use_dynamic_chunk,
                                          self.use_dynamic_left_chunk,
                                          decoding_chunk_size,
                                          self.static_chunk_size,
                                          num_decoding_left_chunks)
    for layer in self.encoders:
      xs, chunk_masks, _, _ = layer(xs, chunk_masks, pos_emb, mask_pad)
    if self.normalize_before:
      xs = self.after_norm(xs)
    # Here we assume the mask is not changed in encoder layers, so just
    # return the masks before encoder layers, and the masks will be used
    # for cross attention with decoder later
    return xs, masks

  def forward_chunk(
      self,
      xs: torch.Tensor,
      offset: int,
      required_cache_size: int,
      att_cache: torch.Tensor = torch.zeros(0, 0, 0, 0),
      cnn_cache: torch.Tensor = torch.zeros(0, 0, 0, 0),
      att_mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """ Forward just one chunk

        Args:
            xs (torch.Tensor): chunk input, with shape (b=1, time, mel-dim),
                where `time == (chunk_size - 1) * subsample_rate + \
                        subsample.right_context + 1`
            offset (int): current offset in encoder output time stamp
            required_cache_size (int): cache size required for next chunk
                compuation
                >=0: actual cache size
                <0: means all history cache is required
            att_cache (torch.Tensor): cache tensor for KEY & VALUE in
                transformer/conformer attention, with shape
                (elayers, head, cache_t1, d_k * 2), where
                `head * d_k == hidden-dim` and
                `cache_t1 == chunk_size * num_decoding_left_chunks`.
            cnn_cache (torch.Tensor): cache tensor for cnn_module in conformer,
                (elayers, b=1, hidden-dim, cache_t2), where
                `cache_t2 == cnn.lorder - 1`

        Returns:
            torch.Tensor: output of current input xs,
                with shape (b=1, chunk_size, hidden-dim).
            torch.Tensor: new attention cache required for next chunk, with
                dynamic shape (elayers, head, ?, d_k * 2)
                depending on required_cache_size.
            torch.Tensor: new conformer cnn cache required for next chunk, with
                same shape as the original cnn_cache.

        """
    assert xs.size(0) == 1
    # tmp_masks is just for interface compatibility
    tmp_masks = torch.ones(1,
                           xs.size(1),
                           device=xs.device,
                           dtype=torch.bool)
    tmp_masks = tmp_masks.unsqueeze(1)
    if self.global_cmvn is not None:
      xs = self.global_cmvn(xs)
    # NOTE(xcsong): Before embed, shape(xs) is (b=1, time, mel-dim)
    xs, pos_emb, _ = self.embed(xs, tmp_masks, offset)
    # NOTE(xcsong): After  embed, shape(xs) is (b=1, chunk_size, hidden-dim)
    elayers, cache_t1 = att_cache.size(0), att_cache.size(2)
    chunk_size = xs.size(1)
    attention_key_size = cache_t1 + chunk_size
    pos_emb = self.embed.position_encoding(
      offset=offset - cache_t1, size=attention_key_size)
    if required_cache_size < 0:
      next_cache_start = 0
    elif required_cache_size == 0:
      next_cache_start = attention_key_size
    else:
      next_cache_start = max(attention_key_size - required_cache_size, 0)
    r_att_cache = []
    r_cnn_cache = []
    for i, layer in enumerate(self.encoders):
      # NOTE(xcsong): Before layer.forward
      #   shape(att_cache[i:i + 1]) is (1, head, cache_t1, d_k * 2),
      #   shape(cnn_cache[i])       is (b=1, hidden-dim, cache_t2)
      xs, _, new_att_cache, new_cnn_cache = layer(
        xs, att_mask, pos_emb,
        att_cache=att_cache[i:i + 1] if elayers > 0 else att_cache,
        cnn_cache=cnn_cache[i] if cnn_cache.size(0) > 0 else cnn_cache
      )
      # NOTE(xcsong): After layer.forward
      #   shape(new_att_cache) is (1, head, attention_key_size, d_k * 2),
      #   shape(new_cnn_cache) is (b=1, hidden-dim, cache_t2)
      r_att_cache.append(new_att_cache[:, :, next_cache_start:, :])
      r_cnn_cache.append(new_cnn_cache.unsqueeze(0))
    if self.normalize_before:
      xs = self.after_norm(xs)

    # NOTE(xcsong): shape(r_att_cache) is (elayers, head, ?, d_k * 2),
    #   ? may be larger than cache_t1, it depends on required_cache_size
    r_att_cache = torch.cat(r_att_cache, dim=0)
    # NOTE(xcsong): shape(r_cnn_cache) is (e, b=1, hidden-dim, cache_t2)
    r_cnn_cache = torch.cat(r_cnn_cache, dim=0)

    return (xs, r_att_cache, r_cnn_cache)

  def forward_chunk_by_chunk(
      self,
      xs: torch.Tensor,
      decoding_chunk_size: int,
      num_decoding_left_chunks: int = -1,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """ Forward input chunk by chunk with chunk_size like a streaming
            fashion

        Here we should pay special attention to computation cache in the
        streaming style forward chunk by chunk. Three things should be taken
        into account for computation in the current network:
            1. transformer/conformer encoder layers output cache
            2. convolution in conformer
            3. convolution in subsampling

        However, we don't implement subsampling cache for:
            1. We can control subsampling module to output the right result by
               overlapping input instead of cache left context, even though it
               wastes some computation, but subsampling only takes a very
               small fraction of computation in the whole model.
            2. Typically, there are several covolution layers with subsampling
               in subsampling module, it is tricky and complicated to do cache
               with different convolution layers with different subsampling
               rate.
            3. Currently, nn.Sequential is used to stack all the convolution
               layers in subsampling, we need to rewrite it to make it work
               with cache, which is not prefered.
        Args:
            xs (torch.Tensor): (1, max_len, dim)
            chunk_size (int): decoding chunk size
        """
    assert decoding_chunk_size > 0
    # The model is trained by static or dynamic chunk
    assert self.static_chunk_size > 0 or self.use_dynamic_chunk
    subsampling = self.embed.subsampling_rate
    context = self.embed.right_context + 1  # Add current frame
    stride = subsampling * decoding_chunk_size
    decoding_window = (decoding_chunk_size - 1) * subsampling + context
    num_frames = xs.size(1)
    att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0), device=xs.device)
    cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0), device=xs.device)
    outputs = []
    offset = 0
    required_cache_size = decoding_chunk_size * num_decoding_left_chunks

    # Feed forward overlap input step by step
    for cur in range(0, num_frames - context + 1, stride):
      end = min(cur + decoding_window, num_frames)
      chunk_xs = xs[:, cur:end, :]
      (y, att_cache, cnn_cache) = self.forward_chunk(
        chunk_xs, offset, required_cache_size, att_cache, cnn_cache)
      outputs.append(y)
      offset += y.size(1)
    ys = torch.cat(outputs, 1)
    masks = torch.ones((1, 1, ys.size(1)), device=ys.device, dtype=torch.bool)
    return ys, masks


class AttentiveStatisticsPooling(torch.nn.Module):
  """This class implements an attentive statistic pooling layer for each channel.
  It returns the concatenated mean and std of the input tensor.

  Arguments
  ---------
  channels: int
      The number of input channels.
  output_channels: int
      The number of output channels.
  """

  def __init__(self, channels, output_channels):
    super().__init__()

    self._eps = 1e-12
    self._linear = Linear(channels * 3, channels)
    self._tanh = torch.nn.Tanh()
    self._conv = Conv1d(
      in_channels=channels, out_channels=channels, kernel_size=1
    )
    self._final_layer = torch.nn.Linear(channels * 2, output_channels,
                                        bias=False)
    logging.info("Init AttentiveStatisticsPooling with {}->{}".format(
      channels, output_channels))
    return

  @staticmethod
  def _compute_statistics(x: torch.Tensor,
                          m: torch.Tensor,
                          eps: float,
                          dim: int = 2):
    mean = (m * x).sum(dim)
    std = torch.sqrt((m * (x - mean.unsqueeze(dim)).pow(2)).sum(dim).clamp(eps))
    return mean, std

  def forward(self, x: torch.Tensor):
    """Calculates mean and std for a batch (input tensor).

    Args:
      x : torch.Tensor
          Tensor of shape [N, L, C].
    """

    x = x.transpose(1, 2)
    L = x.shape[-1]
    lengths = torch.ones(x.shape[0], device=x.device)
    mask = self.length_to_mask(lengths * L, max_len=L, device=x.device)
    mask = mask.unsqueeze(1)
    total = mask.sum(dim=2, keepdim=True).float()

    mean, std = self._compute_statistics(x, mask / total, self._eps)
    mean = mean.unsqueeze(2).repeat(1, 1, L)
    std = std.unsqueeze(2).repeat(1, 1, L)
    attn = torch.cat([x, mean, std], dim=1)
    attn = self._conv(self._tanh(self._linear(
      attn.transpose(1, 2)).transpose(1, 2)))

    attn = attn.masked_fill(mask == 0, float("-inf"))  # Filter out zero-padding
    attn = F.softmax(attn, dim=2)
    mean, std = self._compute_statistics(x, attn, self._eps)
    pooled_stats = self._final_layer(torch.cat((mean, std), dim=1))
    return pooled_stats

  def forward_with_mask(self, x: torch.Tensor,
                        lengths: Optional[torch.Tensor] = None):
    """Calculates mean and std for a batch (input tensor).

    Args:
      x : torch.Tensor
          Tensor of shape [N, C, L].
      lengths:
    """
    L = x.shape[-1]

    if lengths is None:
      lengths = torch.ones(x.shape[0], device=x.device)

    # Make binary mask of shape [N, 1, L]
    mask = self.length_to_mask(lengths * L, max_len=L, device=x.device)
    mask = mask.unsqueeze(1)

    # Expand the temporal context of the pooling layer by allowing the
    # self-attention to look at global properties of the utterance.

    # torch.std is unstable for backward computation
    # https://github.com/pytorch/pytorch/issues/4320
    total = mask.sum(dim=2, keepdim=True).float()
    mean, std = self._compute_statistics(x, mask / total, self._eps)

    mean = mean.unsqueeze(2).repeat(1, 1, L)
    std = std.unsqueeze(2).repeat(1, 1, L)
    attn = torch.cat([x, mean, std], dim=1)

    # Apply layers
    attn = self.conv(self._tanh(self._linear(attn, lengths)))

    # Filter out zero-paddings
    attn = attn.masked_fill(mask == 0, float("-inf"))

    attn = F.softmax(attn, dim=2)
    mean, std = self._compute_statistics(x, attn, self._eps)
    # Append mean and std of the batch
    pooled_stats = torch.cat((mean, std), dim=1)
    pooled_stats = pooled_stats.unsqueeze(2)
    return pooled_stats

  @staticmethod
  def length_to_mask(length: torch.Tensor,
                     max_len: Optional[int] = None,
                     dtype: Optional[torch.dtype] = None,
                     device: Optional[torch.device] = None):
    """Creates a binary mask for each sequence.

    Arguments
    ---------
    length : torch.LongTensor
        Containing the length of each sequence in the batch. Must be 1D.
    max_len : int
        Max length for the mask, also the size of the second dimension.
    dtype : torch.dtype, default: None
        The dtype of the generated mask.
    device: torch.device, default: None
        The device to put the mask variable.

    Returns
    -------
    mask : tensor
        The binary mask.

    Example
    -------
    """
    assert len(length.shape) == 1

    if max_len is None:
      max_len = length.max().long().item()  # using arange to generate mask
    mask = torch.arange(
      max_len, device=length.device, dtype=length.dtype
    ).expand(len(length), max_len) < length.unsqueeze(1)

    if dtype is None:
      dtype = length.dtype

    if device is None:
      device = length.device

    mask = torch.as_tensor(mask, dtype=dtype, device=device)
    return mask


class Model(BasicModel):
  """ Csi Backbone Model:
      Batch-norm
      Conformer-Encoder(x6 or x4)
      Global-Avg-Pool
      Resize and linear
      Bn-neck

  Loss:
    Focal-loss(ce) + Triplet-loss + Center-loss

  Args:
    hp: Dict with all parameters
  """

  def __init__(self, hp: Dict):
    super(BasicModel, self).__init__()
    self._hp = hp

    self._global_cmvn = torch.nn.BatchNorm1d(hp["input_dim"])
    self._encoder = ConformerEncoder(
      input_size=hp["input_dim"],
      output_size=hp["encoder"]["output_dims"],
      linear_units=hp["encoder"]["attention_dim"],
      num_blocks=hp["encoder"]["num_blocks"])

    if hp["encoder"]["output_dims"] != hp["embed_dim"]:
      self._embed_lo = torch.nn.Linear(hp["encoder"]["output_dims"],
                                       hp["embed_dim"])
    else:
      self._embed_lo = None

    # Bottleneck
    self._bottleneck = torch.nn.BatchNorm1d(hp["embed_dim"])
    self._bottleneck.bias.requires_grad_(False)  # no shift

    self._pool_layer = AttentiveStatisticsPooling(
      hp["embed_dim"], output_channels=hp["embed_dim"])
    self._ce_layer = torch.nn.Linear(
      hp["embed_dim"], hp["ce"]["output_dims"], bias=False)

    # Loss
    if "alpha" in hp["ce"].keys():
      alpha = np.load(hp["ce"]["alpha"])
      alpha = 1.0 / (alpha + 1)
      alpha = alpha / np.sum(alpha)
      logging.warning("use alpha with {}".format(len(alpha)))
    else:
      alpha = None
      logging.warning("Not use alpha")

    self._ce_loss = FocalLoss(alpha=alpha, gamma=self._hp["ce"]["gamma"],
                              num_cls=self._hp["ce"]["output_dims"])
    self._triplet_loss = HardTripletLoss(margin=hp["triplet"]["margin"])
    self._center_loss = CenterLoss(num_classes=self._hp["ce"]["output_dims"],
                                   feat_dim=hp["embed_dim"], use_gpu=torch.cuda.is_available())
    logging.info(
      "Model size: {:.3f}M \n".format(self.model_size() / 1000 / 1000))
    return

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """feat[b, frame_size, feat_size] -> embed[b, embed_dim]"""

    x = self._global_cmvn(x.transpose(1, 2)).transpose(1, 2)
    xs_lens = torch.full(
      [x.size(0)], fill_value=x.size(1), dtype=torch.long).to(x.device).long()
    x, _ = self._encoder(x, xs_lens=xs_lens, decoding_chunk_size=-1)
    x = self._pool_layer(x)
    return x

  @torch.jit.ignore
  def compute_loss(self, anchor: torch.Tensor, label: torch.Tensor) -> Tuple[
    torch.Tensor, Dict]:
    """compute ce and triplet loss"""
    f_t = self.forward(anchor)
    # print("f_t", f_t)

    f_i = self._bottleneck(f_t)
    ce_pred = self._ce_layer(f_i)
    # print("ce_pred", ce_pred)
    ce_loss = self._ce_loss(ce_pred, label)
    loss = ce_loss * self._hp["ce"]["weight"]
    loss_dict = {"ce_loss": ce_loss}

    tri_weight = self._hp["triplet"]["weight"]
    if tri_weight > 0.01:
      tri_loss = self._triplet_loss(f_t, label)
      loss = loss + tri_loss * tri_weight
      loss_dict.update({"tri_loss": tri_loss})

    cen_weight = self._hp["center"]["weight"]
    if cen_weight > 0.0:
      cen_loss = self._center_loss(f_t, label)
      loss = loss + cen_loss * cen_weight
      loss_dict.update({"cen_loss": cen_loss})
    return loss, loss_dict

  @torch.jit.ignore
  def inference(self, feat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    with torch.no_grad():
      embed = self.forward(feat)
      embed_ce = self._ce_layer(embed)
    return embed, embed_ce

  @torch.jit.export
  def compute_embed(self, feat: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
      embed = self.forward(feat)
    return embed


In [ ]:
class AudioPreprocessor:
    """Tiền xử lý âm thanh: Resample về 16,000 Hz, chuẩn hóa biên độ và cắt Sliding Window 10s / hop 3s."""
    def __init__(self, sample_rate: int = 16000, chunk_duration: float = 10.0, hop_duration: float = 3.0):
        self.sample_rate = sample_rate
        self.chunk_duration = chunk_duration
        self.hop_duration = hop_duration
        self.chunk_samples = int(chunk_duration * sample_rate)
        self.hop_samples = int(hop_duration * sample_rate)

    def load_audio(self, audio_path: str) -> np.ndarray:
        """Nạp và chuẩn hóa âm thanh, không nuốt lỗi âm thầm."""
        if not os.path.exists(audio_path):
            raise FileNotFoundError(f"Không tìm thấy file âm thanh: {audio_path}")
            
        if audio_path.lower().endswith('.silent'):
            return np.zeros(self.chunk_samples, dtype=np.float32)

        try:
            if audio_path.lower().endswith('.mp3'):
                y, sr = librosa.load(audio_path, sr=self.sample_rate, duration=None)
            else:
                y, sr = sf.read(audio_path, always_2d=False, dtype='float32')
                if len(y.shape) > 1:
                    y = np.mean(y, axis=1)
                if sr != self.sample_rate:
                    y = librosa.resample(y, orig_sr=sr, target_sr=self.sample_rate)
        except Exception as e:
            # Fallback thu doc bang librosa neu soundfile gap loi dinh dang wav
            try:
                y, sr = librosa.load(audio_path, sr=self.sample_rate, duration=None)
            except Exception as final_e:
                print(f"[!] Canh bao: Khong the doc file {audio_path}: {final_e}")
                raise RuntimeError(f"Không thể đọc file âm thanh: {audio_path}") from final_e

        y = np.asarray(y, dtype=np.float32).reshape(-1)
        if y.size == 0:
            raise ValueError(f"File audio rong: {audio_path}")
        if not np.isfinite(y).all():
            raise ValueError(f"Audio chua NaN/Inf: {audio_path}")
        max_val = float(np.max(np.abs(y)))
        if max_val > 1e-6:
            y = y / max_val
        return y

    def slice_into_chunks(self, y: np.ndarray) -> list:
        """Cắt âm thanh thành danh sách các đoạn 10 giây (hop 3s). Trả về ít nhất 1 đoạn."""
        if len(y) <= self.chunk_samples:
            return [np.pad(y, (0, self.chunk_samples - len(y)), mode='constant')]
        chunks = []
        start = 0
        for start in range(0, len(y) - self.chunk_samples + 1, self.hop_samples):
            chunks.append(y[start : start + self.chunk_samples])
        last_chunk_end = start + self.chunk_samples
        remaining_unseen = len(y) - last_chunk_end
        if remaining_unseen > (self.chunk_samples // 3):
            final_chunk = y[-self.chunk_samples:]
            if not np.array_equal(final_chunk, chunks[-1]):
                chunks.append(final_chunk)
        return chunks if chunks else [np.pad(y, (0, max(0, self.chunk_samples - len(y))), mode='constant')[:self.chunk_samples]]

In [ ]:
import scipy.sparse

def shorter(feat: np.ndarray, mean_size: int = 3) -> np.ndarray:
    """Nén trục thời gian của ma trận đặc trưng CQT theo bước mean_size (CoverHunter chuẩn)."""
    if mean_size <= 1:
        return feat
    cqt = feat.T
    height, length = cqt.shape
    new_len = int(length / mean_size)
    new_cqt = np.zeros((height, new_len), dtype=np.float32)
    for i in range(new_len):
        new_cqt[:, i] = cqt[:, i * mean_size : (i + 1) * mean_size].mean(axis=1)
    return new_cqt.T

class PyCqt:
    """Thuật toán CQT Sparse Kernel chính thức của CoverHunter (Tsinghua/MIREX)."""
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = None):
        self._hop_size = hop_size
        self._sample_rate = sample_rate
        if not max_freq:
            max_freq = sample_rate // 2
        self._kernel = self._compute_cqt_kernel(sample_rate, octave_resolution, min_freq, max_freq)

    @staticmethod
    def _compute_cqt_kernel(sampling_frequency: int, octave_resolution: int, minimum_frequency: float, maximum_frequency: float):
        quality_factor = 1.0 / (pow(2.0, 1.0 / octave_resolution) - 1.0)
        number_frequencies = round(octave_resolution * np.log2(maximum_frequency / minimum_frequency))
        fft_length = int(pow(2, np.ceil(np.log2(quality_factor * sampling_frequency / minimum_frequency))))
        cqt_kernel = np.zeros((number_frequencies, fft_length), dtype=complex)

        for i in range(number_frequencies):
            frequency_value = minimum_frequency * pow(2.0, i / octave_resolution)
            window_length = 2 * round(quality_factor * sampling_frequency / frequency_value / 2.0) + 1
            temporal_kernel = (np.hamming(window_length)
                               * np.exp(2.0 * np.pi * 1j * quality_factor
                                        * np.arange(-(window_length - 1) / 2.0, (window_length - 1) / 2.0 + 1) / window_length)
                               / window_length)
            pad_width = int((fft_length - window_length + 1) / 2)
            cqt_kernel[i, pad_width : pad_width + window_length] = temporal_kernel

        cqt_kernel = np.fft.fft(cqt_kernel, axis=1)
        cqt_kernel[np.absolute(cqt_kernel) < 0.01] = 0
        cqt_kernel = scipy.sparse.csr_matrix(cqt_kernel)
        cqt_kernel = np.conjugate(cqt_kernel) / fft_length
        return cqt_kernel

    @staticmethod
    def _compute_cqt_spec(audio_signal: np.ndarray, sampling_frequency: int, time_resolution: int, cqt_kernel):
        step_length = round(sampling_frequency / time_resolution)
        number_times = int(np.floor(len(audio_signal) / step_length))
        number_frequencies, fft_length = np.shape(cqt_kernel)

        pad_left = int(np.ceil((fft_length - step_length) / 2.0))
        pad_right = int(np.floor((fft_length - step_length) / 2.0))
        audio_signal = np.pad(audio_signal, (pad_left, pad_right), "constant", constant_values=(0, 0))

        cqt_spectrogram = np.zeros((number_frequencies, number_times), dtype=np.float32)
        idx = 0
        for j in range(number_times):
            cqt_spectrogram[:, j] = np.absolute(cqt_kernel * np.fft.fft(audio_signal[idx : idx + fft_length]))
            idx += step_length
        return cqt_spectrogram

    def compute_cqt(self, signal_float: np.ndarray, feat_dim_first: bool = False) -> np.ndarray:
        time_resolution = int(round(1.0 / self._hop_size))
        cqt_spectrogram = self._compute_cqt_spec(signal_float, self._sample_rate, time_resolution, self._kernel)
        magnitude = np.maximum(cqt_spectrogram, 1e-9)
        ref_value = max(float(np.max(magnitude)), 1e-4)
        cqt_spectrogram = 20.0 * np.log10(magnitude) - 20.0 * np.log10(ref_value)
        cqt_spectrogram = np.clip(cqt_spectrogram, -80.0, 0.0)
        if not feat_dim_first:
            cqt_spectrogram = cqt_spectrogram.T
        return cqt_spectrogram

class CoverHunterCQT:
    """
    Trích xuất đặc trưng CQT chuẩn CoverHunter:
    - 96 bins (12 bins/octave, 8 octaves từ 32 Hz đến 8000 Hz)
    - hop_size: 0.04s (25 fps), nén thời gian shorter(mean_size=3)
    - Chuẩn hóa dB theo biên độ cực đại: 20 * log10(cqt / max_cqt)
    """
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = 8000.0, mean_size: int = 3):
        self.sample_rate = sample_rate
        self.mean_size = mean_size
        self.py_cqt = PyCqt(
            sample_rate=sample_rate,
            hop_size=hop_size,
            octave_resolution=octave_resolution,
            min_freq=min_freq,
            max_freq=max_freq
        )

    def extract(self, audio: np.ndarray) -> torch.Tensor:
        if isinstance(audio, torch.Tensor):
            audio = audio.detach().cpu().numpy()
        feat = self.py_cqt.compute_cqt(audio, feat_dim_first=False) # (T, 96)
        if self.mean_size > 1:
            feat = shorter(feat, mean_size=self.mean_size)
        return torch.from_numpy(feat).float()

In [ ]:
class CQTExtractor:
    """Trích xuất Constant-Q Transform (CQT) hỗ trợ cả chế độ Fixed-Length (83 frames) và Variable-Length."""
    def __init__(self, sample_rate: int = 16000, n_bins: int = 96, bins_per_octave: int = 12, hop_size: float = 0.04, mean_size: int = 3, target_frames: int = 83):
        self.sample_rate = sample_rate
        self.n_bins = n_bins
        self.bins_per_octave = bins_per_octave
        self.hop_size = hop_size
        self.mean_size = mean_size
        self.target_frames = target_frames
        self.cqt_impl = CoverHunterCQT(
            sample_rate=sample_rate,
            hop_size=hop_size,
            octave_resolution=bins_per_octave,
            min_freq=32.0,
            max_freq=8000.0,
            mean_size=mean_size
        )

    def extract(self, y: np.ndarray, variable_length: bool = False) -> torch.Tensor:
        """
        Trích xuất tensor CQT từ tín hiệu âm thanh 1D.
        - variable_length=False: Căn chỉnh về đúng [1, 96, target_frames] (mặc định 83 frames ~ 10s).
        - variable_length=True: Giữ nguyên độ dài thời gian thực tế [1, 96, T] phục vụ Stage 2 DTW.
        """
        feat = self.cqt_impl.extract(y)  # (T, 96)
        tensor = feat.T.unsqueeze(0)      # (1, 96, T)

        if not variable_length and self.target_frames is not None and self.target_frames > 0:
            curr_t = tensor.shape[-1]
            min_val = tensor.min()
            if curr_t < self.target_frames:
                tensor = F.pad(tensor, (0, self.target_frames - curr_t), mode='constant', value=min_val.item())
            elif curr_t > self.target_frames:
                tensor = tensor[:, :, :self.target_frames]
        return tensor

SAMPLE_RATE = 16000
CHUNK_DURATION_SEC = 10.0
CHUNK_HOP_DURATION_SEC = 3.0

prep = AudioPreprocessor(sample_rate=SAMPLE_RATE, chunk_duration=CHUNK_DURATION_SEC, hop_duration=CHUNK_HOP_DURATION_SEC)
cqt_ext = CQTExtractor(sample_rate=16000, n_bins=96, bins_per_octave=12, hop_size=0.04, mean_size=3, target_frames=83)
print("[+] Khởi tạo thành công AudioPreprocessor và CoverHunter CQT Standalone (96 Bins, 83 Frames, dB Scale)!")

In [ ]:
def apply_cqt_pitch_shift(cqt: torch.Tensor, shift_bins: int) -> torch.Tensor:
    """
    Dịch chuyển cao độ trên ma trận CQT bằng cách tịnh tiến trục tần số (dim -2).
    - shift_bins > 0: Dịch lên cao độ cao hơn (tăng semitone).
    - shift_bins < 0: Dịch xuống cao độ thấp hơn (giảm semitone).
    - Điền các bin trống bằng cqt.min() (mức sàn im lặng), KHÔNG điền 0.0 (vốn là 0 dB cực đại).
    """
    if shift_bins == 0:
        return cqt

    min_val = cqt.min()
    shifted = torch.full_like(cqt, fill_value=min_val)
    num_bins = cqt.shape[-2]

    if shift_bins > 0:
        if shift_bins < num_bins:
            shifted[..., shift_bins:, :] = cqt[..., :num_bins - shift_bins, :]
    else:
        abs_shift = abs(shift_bins)
        if abs_shift < num_bins:
            shifted[..., :num_bins - abs_shift, :] = cqt[..., abs_shift:, :]

    return shifted

In [ ]:
class CQTPitchShiftAugmentation:
    """
    Tăng cường dữ liệu dịch tone ngẫu nhiên (Full-Octave Pitch-Shift Data Augmentation).
    Dịch ngẫu nhiên trong dải [-max_shift_bins, +max_shift_bins] nửa cung (semitones).
    Mặc định dải ±12 semitones bao trọn 1 quãng 8 (chuyển đổi hoàn hảo giữa giọng Nam và Nữ).
    """
    def __init__(self, max_shift_bins: int = 12, prob: float = 0.8):
        self.max_shift_bins = max_shift_bins
        self.prob = prob

    def __call__(self, cqt_tensor: torch.Tensor) -> torch.Tensor:
        if random.random() > self.prob:
            return cqt_tensor
        shift = random.randint(-self.max_shift_bins, self.max_shift_bins)
        return apply_cqt_pitch_shift(cqt_tensor, shift)

In [ ]:
class SpectrogramTempoAndMaskDistortion:
    """Nonlinear time warp plus SpecAugment masks; the CQT frequency axis is never warped."""
    def __init__(self, time_scale_range=(0.5, 1.5), max_mask_t=0, max_mask_f=0, prob=0.6, knots=6):
        self.scale_range = time_scale_range
        self.max_mask_t = int(max_mask_t)
        self.max_mask_f = int(max_mask_f)
        self.prob = prob
        self.knots = knots
        if not 0 < time_scale_range[0] <= time_scale_range[1] or knots < 2:
            raise ValueError('Invalid distortion parameters')
        if self.max_mask_t < 0 or self.max_mask_f < 0:
            raise ValueError('Mask widths must be non-negative')

    def _axis(self, size, batch, device, dtype):
        increments = torch.empty(batch, 1, self.knots, device=device, dtype=dtype).uniform_(*self.scale_range)
        positions = torch.cat([torch.zeros(batch, 1, 1, device=device, dtype=dtype), increments.cumsum(-1)], -1)
        positions = 2 * positions / positions[..., -1:] - 1
        return F.interpolate(positions, size=size, mode='linear', align_corners=True)[:, 0]

    @staticmethod
    def _mask_axis(x, axis, max_width):
        if max_width <= 0 or x.shape[axis] <= 1:
            return
        width = random.randint(1, min(max_width, x.shape[axis] - 1))
        start = random.randint(0, x.shape[axis] - width)
        slices = [slice(None)] * x.ndim
        slices[axis] = slice(start, start + width)
        fill_value = x.amin(dim=(-2, -1), keepdim=True)
        x[tuple(slices)] = fill_value

    def __call__(self, spec):
        if random.random() >= self.prob:
            return spec
        was_unbatched = spec.ndim == 3
        x = spec.unsqueeze(0) if was_unbatched else spec
        b, _, h, w = x.shape
        tx = self._axis(w, b, x.device, x.dtype)
        fy = torch.linspace(-1, 1, h, device=x.device, dtype=x.dtype).expand(b, h)
        grid = torch.stack([tx[:, None, :].expand(b, h, w), fy[:, :, None].expand(b, h, w)], -1)
        out = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True).clone()
        self._mask_axis(out, axis=-1, max_width=self.max_mask_t)
        self._mask_axis(out, axis=-2, max_width=self.max_mask_f)
        return out.squeeze(0) if was_unbatched else out


## Bounded audio/CQT cache


In [ ]:
from collections import OrderedDict
import hashlib

class BoundedCQTCache(OrderedDict):
    def __init__(self, max_mb):
        super().__init__()
        self.limit = int(max_mb * 1024**2)
        self.bytes_used = 0

    @staticmethod
    def size(value):
        return sum(t.numel() * t.element_size() for t in value)

    def __getitem__(self, key):
        value = super().__getitem__(key)
        self.move_to_end(key)
        return value

    def __setitem__(self, key, value):
        if key in self:
            self.bytes_used -= self.size(super().__getitem__(key))
            super().__delitem__(key)
        size = self.size(value)
        if size > self.limit:
            return
        while self and self.bytes_used + size > self.limit:
            _, old = self.popitem(last=False)
            self.bytes_used -= self.size(old)
        super().__setitem__(key, value)
        self.bytes_used += size

GLOBAL_CQT_RAM_CACHE = BoundedCQTCache(LOCAL_RAM_CACHE_MB)
CQT_CACHE_DIR = STAGE2_ROOT / 'cache' / 'cqt_timegrid_v4_16k_10s_hop3_96x83'
CQT_CACHE_DIR.mkdir(parents=True, exist_ok=True)


class ByteHumCachedTripletDataset(Dataset):
    """
    Tập dữ liệu sinh bộ ba Triplet (Anchor, Positive, Negative) với cơ chế In-RAM CQT Caching:
    - Lưu danh sách tất cả các chunk CQT của mỗi file audio vào RAM, khắc phục lỗi chỉ lấy chunks[0].
    - Khi training: ngẫu nhiên chọn 1 chunk từ danh sách chunks cho mỗi epoch để học đa dạng vị trí.
    - Khi validation: lấy cố định chunks[0] để kết quả đánh giá mang tính xác định (deterministic).
    - Áp dụng Pitch-Shift (±12 semitones) và Tempo Distortion ngẫu nhiên động trên CQT.
    """
    def __init__(
        self,
        split_dir: str,
        audio_preprocessor: AudioPreprocessor,
        cqt_extractor: CQTExtractor,
        tempo_distortion: Optional[SpectrogramTempoAndMaskDistortion] = None,
        pitch_shifter: Optional[CQTPitchShiftAugmentation] = None,
        is_training: bool = True,
        song_to_idx: Optional[Dict[str, int]] = None,
        vocal_ratio: float = 0.80
    ):
        self.split_dir = split_dir
        self.audio_preprocessor = audio_preprocessor
        self.cqt_extractor = cqt_extractor
        self.tempo_distortion = tempo_distortion
        self.pitch_shifter = pitch_shifter
        self.is_training = is_training
        self.vocal_ratio = vocal_ratio

        if not split_dir or not os.path.exists(split_dir):
            raise FileNotFoundError(f"Không tìm thấy thư mục: {split_dir}")

        self.songs = sorted([s for s in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, s))])
        if song_to_idx is not None:
            self.song_to_idx = song_to_idx
        else:
            self.song_to_idx = {s: i for i, s in enumerate(self.songs)}

        self.song_data = {}
        self.humming_samples = []

        for s in self.songs:
            s_path = os.path.join(split_dir, s)
            frags = sorted([f for f in os.listdir(s_path) if os.path.isdir(os.path.join(s_path, f))])
            self.song_data[s] = {}
            for f in frags:
                f_path = os.path.join(s_path, f)
                audio_extensions = ('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.silent')
                audio_files = sorted(w for w in os.listdir(f_path) if w.lower().endswith(audio_extensions))
                refs = [os.path.join(f_path, w) for w in audio_files if w == 'reference.wav']
                vocals = [os.path.join(f_path, w) for w in audio_files if w.startswith('reference_vocal')]
                hums = [os.path.join(f_path, w) for w in audio_files if not w.startswith('reference')]
                if refs or vocals or hums:
                    self.song_data[s][f] = {'refs': refs, 'vocals': vocals, 'hums': hums}
                    for h in hums:
                        self.humming_samples.append((s, f, h))

    def __len__(self) -> int:
        return len(self.humming_samples)

    def _get_raw_cqts(self, path: str) -> List[torch.Tensor]:
        """Lấy danh sách tất cả các tensor CQT chunks từ RAM Cache, nếu chưa có thì trích xuất toàn bộ."""
        if path in GLOBAL_CQT_RAM_CACHE:
            return GLOBAL_CQT_RAM_CACHE[path]
        source = Path(path).resolve()
        stat = source.stat()
        fingerprint = f'{source}|{stat.st_size}|{stat.st_mtime_ns}|v4_16k_10s_hop3_96x83'
        cache_path = CQT_CACHE_DIR / (hashlib.sha256(fingerprint.encode()).hexdigest() + '.pt')
        if cache_path.is_file():
            cqts = torch.load(cache_path, map_location='cpu', weights_only=True)
            if not cqts or any(t.shape != (1, 96, 83) or not torch.isfinite(t).all() for t in cqts):
                raise RuntimeError(f'Invalid CQT cache; remove this file: {cache_path}')
            GLOBAL_CQT_RAM_CACHE[path] = cqts
            return cqts
        try:
            y = self.audio_preprocessor.load_audio(path)
            chunks = self.audio_preprocessor.slice_into_chunks(y)
            cqts = [self.cqt_extractor.extract(chk, variable_length=False) for chk in chunks]
        except Exception as e:
            raise RuntimeError(f"Không thể tạo CQT cho {path}") from e
        temp_path = cache_path.with_suffix('.tmp')
        torch.save(cqts, temp_path)
        os.replace(temp_path, cache_path)
        GLOBAL_CQT_RAM_CACHE[path] = cqts
        return cqts

    def _sample_chunk(self, path: str) -> torch.Tensor:
        """Chọn 1 chunk CQT: ngẫu nhiên khi train, cố định chunk 0 khi val."""
        cqts = self._get_raw_cqts(path)
        if self.is_training:
            return random.choice(cqts).clone()
        return cqts[0].clone()

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        song_a, frag_a, hum_a = self.humming_samples[idx]
        
        # 1. Anchor: Tiếng ngân nga (Domain 1.0)
        cqt_a = self._sample_chunk(hum_a)
        if self.is_training:
            if self.tempo_distortion: cqt_a = self.tempo_distortion(cqt_a)
            if self.pitch_shifter: cqt_a = self.pitch_shifter(cqt_a)

        # 2. Positive: Bản thu gốc hoặc vocal tách rời
        pool = self.song_data[song_a][frag_a]
        if self.is_training and pool['vocals'] and random.random() < self.vocal_ratio:
            pos_path, domain_p = random.choice(pool['vocals']), 0.0
        elif pool['refs']:
            pos_path, domain_p = (random.choice(pool['refs']) if self.is_training else pool['refs'][0]), 0.0
        elif pool['vocals']:
            pos_path, domain_p = (random.choice(pool['vocals']) if self.is_training else pool['vocals'][0]), 0.0
        else:
            cands = sorted(h for h in pool['hums'] if h != hum_a)
            if not cands:
                raise RuntimeError(f"Thiếu positive phù hợp cho {song_a}/{frag_a}")
            pos_path, domain_p = (random.choice(cands) if self.is_training else cands[0]), 1.0
            
        cqt_p = self._sample_chunk(pos_path)
        if self.is_training and domain_p == 1.0:
            if self.pitch_shifter: cqt_p = self.pitch_shifter(cqt_p)

        # 3. Negative: Bản thu của bài hát khác hoàn toàn
        neg_song_candidates = [s for s in self.songs if s != song_a and self.song_data[s]]
        if not neg_song_candidates:
            raise RuntimeError(f"Không có negative hợp lệ cho {song_a}")
        if self.is_training:
            neg_song = random.choice(neg_song_candidates)
            neg_frag = random.choice(sorted(self.song_data[neg_song]))
        else:
            neg_song = neg_song_candidates[idx % len(neg_song_candidates)]
            neg_fragments = sorted(self.song_data[neg_song])
            neg_frag = neg_fragments[idx % len(neg_fragments)]
        neg_pool = self.song_data[neg_song][neg_frag]
        if self.is_training:
            if random.random() > 0.5 and neg_pool['refs']:
                neg_path, domain_n = random.choice(neg_pool['refs']), 0.0
            elif neg_pool['hums']:
                neg_path, domain_n = random.choice(neg_pool['hums']), 1.0
            elif neg_pool['vocals']:
                neg_path, domain_n = random.choice(neg_pool['vocals']), 0.0
            else:
                raise RuntimeError(f"Thiếu negative audio cho {neg_song}/{neg_frag}")
        elif neg_pool['refs']:
            neg_path, domain_n = neg_pool['refs'][0], 0.0
        elif neg_pool['hums']:
            neg_path, domain_n = neg_pool['hums'][0], 1.0
        elif neg_pool['vocals']:
            neg_path, domain_n = neg_pool['vocals'][0], 0.0
        else:
            raise RuntimeError(f"Thiếu negative audio cho {neg_song}/{neg_frag}")
            
        cqt_n = self._sample_chunk(neg_path)

        song_label = self.song_to_idx[song_a]
        neg_song_label = self.song_to_idx[neg_song]

        return {
            'anchor': cqt_a,
            'positive': cqt_p,
            'negative': cqt_n,
            'song_idx': torch.tensor(song_label, dtype=torch.long),
            'negative_song_idx': torch.tensor(neg_song_label, dtype=torch.long),
            'domain_anchor': torch.tensor([1.0], dtype=torch.float),
            'domain_positive': torch.tensor([domain_p], dtype=torch.float),
            'domain_negative': torch.tensor([domain_n], dtype=torch.float)
        }

In [ ]:
from torch.autograd import Function

class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambda_val: float):
        ctx.lambda_val = lambda_val
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return grad_output.neg() * ctx.lambda_val, None

class GradientReversalLayer(nn.Module):
    """Lớp đảo ngược gradient với hệ số lambda điều chỉnh động theo epoch."""
    def __init__(self, lambda_val: float = 0.2):
        super(GradientReversalLayer, self).__init__()
        self.lambda_val = lambda_val

    def set_lambda(self, lambda_val: float):
        self.lambda_val = lambda_val

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return GradientReversalFunction.apply(x, self.lambda_val)

In [ ]:
class LocalDomainClassifier(nn.Module):
    """
    Bộ phân loại miền cục bộ (Local Domain Classifier) theo kiến trúc ByteHum:
    - Nhận đầu vào: Chuỗi đặc trưng cục bộ x [B, T, 128] từ Conformer encoder.
    - Áp dụng các tầng tích chập 1D theo trục thời gian để học đặc trưng miền cục bộ.
    - Dự đoán phân phối nhãn miền (0: Vocal phong thu, 1: Tiếng humming) trên từng phân đoạn thời gian.
    """
    def __init__(self, feature_dim: int = 128, hidden_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(feature_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GroupNorm(1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GroupNorm(1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, 128] -> transpose thành [B, 128, T] cho Conv1D
        x_t = x.transpose(1, 2)
        feat = self.conv(x_t) # [B, hidden_dim, T]
        # Frame-level logits: [B, T]
        frame_logits = self.fc(feat.transpose(1, 2)).squeeze(-1)
        # Tổng hợp logits qua trục thời gian T
        return frame_logits.mean(dim=-1) # [B]

In [ ]:
def build_coverhunter_model(ckpt_path: str = None, device: str = "cpu", allow_random_init: bool = False):
    """
    Khởi tạo mô hình CoverHunter Conformer chính thức (9.91M tham số, 6 khối Conformer, Attention Pooling 128-D).
    Nạp trọn vẹn 265/265 tensor từ checkpoint SHS100K pre-trained.
    """
    # Cấu hình siêu tham số mặc định chuẩn xác của CoverHunter SHS100K
    hp = {
        'input_dim': 96, 'embed_dim': 128,
        'encoder': {'output_dims': 128, 'num_blocks': 6, 'attention_dim': 256},
        'pool_type': 'attention',
        'ce': {'output_dims': 30000, 'weight': 1.0, 'gamma': 2},
        'triplet': {'margin': 0.3, 'weight': 0.1},
        'center': {'weight': 0.0}
    }

    ch_model = Model(hp)

    if ckpt_path and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state_dict = ckpt.get("generator", ckpt.get("model_state_dict", ckpt))
        ch_model.load_state_dict(state_dict)
        total_p = sum(p.numel() for p in ch_model.parameters())
        print(f"[+] Nạp thành công 265/265 tensors ({total_p:,} params) từ CoverHunter Pretrained:\n    -> {ckpt_path}")
    elif allow_random_init:
        print("[*] Khoi tao backbone tam thoi; checkpoint fine-tuned se duoc nap ngay sau do.")
    else:
        raise FileNotFoundError(f"Bat buoc phai co checkpoint Stage 1 SHS100K: {ckpt_path}")

    ch_model = ch_model.to(device)
    return ch_model

In [ ]:
class LocalAlignmentClassifier(nn.Module):
    """
    Local Alignment Classifier (LAC) theo kiến trúc ByteHum (ICASSP 2024):
    - Trọng số Prototype: W in R^{K x L x C}, trong đó K là số lớp bài hát (train songs),
      L là số phân đoạn prototype cục bộ (mặc định L=4), C=128.
    - Điểm tương đồng MaxMean: S(X, W_k) = (1/L) * sum_{l=1}^L max_{t=1}^T cos(x_t, w_{k,l}).
    - Cho phép căn chỉnh cục bộ bất biến thời gian và trôi dạt nhịp điệu giữa Humming và Vocal.
    """
    def __init__(self, num_classes: int = 210, feature_dim: int = 128, num_prototypes: int = 4):
        super().__init__()
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.num_prototypes = num_prototypes
        # Khởi tạo ma trận prototype W [K, L, C]
        self.prototypes = nn.Parameter(torch.randn(num_classes, num_prototypes, feature_dim) * 0.05)
        self.temperature = nn.Parameter(torch.tensor(10.0))

    def forward(self, local_feat: torch.Tensor) -> torch.Tensor:
        # local_feat: [B, T, 128]
        # Chuẩn hóa L2 trên trục đặc trưng 128-D
        x_norm = F.normalize(local_feat, p=2, dim=-1) # [B, T, C]
        w_norm = F.normalize(self.prototypes, p=2, dim=-1) # [K, L, C]

        # Tính cosine similarity giữa toàn bộ T frames và K*L prototypes qua phép nhân ma trận:
        w_flat = w_norm.view(-1, self.feature_dim).t() # [C, K*L]
        sim = torch.matmul(x_norm, w_flat) # [B, T, K*L]
        B, T, _ = sim.shape
        sim = sim.view(B, T, self.num_classes, self.num_prototypes) # [B, T, K, L]

        # Phep toan MaxMean: Max theo truc thoi gian T, Mean theo so prototypes L
        max_t, _ = sim.max(dim=1) # [B, K, L]
        logits = max_t.mean(dim=-1) # [B, K]
        return logits * self.temperature.clamp(min=1.0, max=100.0)

class ConformerByteHumModel(nn.Module):
    """
    MÔ HÌNH HOÀN CHỈNH CONFORMER-BYTEHUM CHO QBH DOMAIN ADAPTATION:
    - Backbone: Conformer (6 khối, Conv2d Subsampling 4x -> T_enc=20 frames cho 10s audio)
    - Output 1: global_emb [B, 128] (qua Attentive Statistics Pooling + L2 Normalize)
    - Output 2: local_feat [B, T=20, 128] (chuỗi đặc trưng cục bộ trước pooling)
    - Output 3: lac_logits [B, K] (Local Alignment Classifier phân loại bài hát qua MaxMean)
    - Output 4: domain_logits [B] (Local Domain Classifier phân biệt miền Vocal vs Humming)
    """
    def __init__(self, ckpt_path: str = None, input_dim: int = 96, embedding_dim: int = 128, num_classes: int = 210, lambda_grl: float = 0.2, allow_random_init: bool = False):
        super().__init__()
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        if embedding_dim not in (128, 256, 512):
            raise ValueError('embedding_dim must be 128, 256 or 512')
        self.local_projection = nn.Identity() if embedding_dim == 128 else nn.Linear(128, embedding_dim)
        self.global_projection = nn.Identity() if embedding_dim == 128 else nn.Linear(128, embedding_dim)

        # Nạp backbone Conformer pre-trained từ CoverHunter Stage 1
        self.feature_extractor = build_coverhunter_model(ckpt_path=ckpt_path, allow_random_init=allow_random_init)
        
        # Đóng băng các tầng phụ không dùng của Stage 1 (ce_layer, center_loss)
        if hasattr(self.feature_extractor, '_ce_layer'):
            for p in self.feature_extractor._ce_layer.parameters(): p.requires_grad = False
        if hasattr(self.feature_extractor, '_center_loss'):
            for p in self.feature_extractor._center_loss.parameters(): p.requires_grad = False

        self.grl = GradientReversalLayer(lambda_val=lambda_grl)
        self.domain_classifier = LocalDomainClassifier(feature_dim=embedding_dim, hidden_dim=64)
        self.lac = LocalAlignmentClassifier(num_classes=num_classes, feature_dim=embedding_dim, num_prototypes=4)

    def train(self, mode: bool = True):
        super().train(mode)
        # Preserve the SHS100K CMVN statistics while fine-tuning on humming audio.
        if mode and hasattr(self.feature_extractor, '_global_cmvn'):
            self.feature_extractor._global_cmvn.eval()
        return self

    def set_lambda(self, lambda_val: float):
        self.grl.set_lambda(lambda_val)

    def _prepare_input(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 4: x = x.squeeze(1)
        if x.dim() == 3:
            if x.shape[1] == 96 and x.shape[2] != 96: x = x.transpose(1, 2)
        elif x.dim() == 2:
            if x.shape[0] == 96 and x.shape[1] != 96: x = x.t().unsqueeze(0)
            else: x = x.unsqueeze(0)
        return x

    def extract_backbone_features(self, x: torch.Tensor) -> torch.Tensor:
        """Trích xuất chuỗi đặc trưng cục bộ [B, T=20, 128] từ Conformer encoder trước ASP pooling."""
        x_prep = self._prepare_input(x)
        x_cmvn = self.feature_extractor._global_cmvn(x_prep.transpose(1, 2)).transpose(1, 2)
        xs_lens = torch.full([x_cmvn.size(0)], fill_value=x_cmvn.size(1), dtype=torch.long, device=x_cmvn.device)
        x_enc, _ = self.feature_extractor._encoder(x_cmvn, xs_lens=xs_lens, decoding_chunk_size=-1)
        return x_enc # [B, T=20, 128]

    def extract_local_features(self, x):
        return self.local_projection(self.extract_backbone_features(x))

    def extract_global_embedding(self, local_feat: torch.Tensor) -> torch.Tensor:
        """Chiếu chuỗi đặc trưng cục bộ qua Attentive Statistics Pooling thành Global 128-D Embedding."""
        emb = self.feature_extractor._pool_layer(local_feat)
        return F.normalize(self.global_projection(emb), p=2, dim=-1)

    def extract_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """Hàm tiện ích trích xuất trực tiếp Global Embedding 128-D từ phổ CQT đầu vào."""
        return self.extract_global_embedding(self.extract_backbone_features(x))

    def forward(self, x: torch.Tensor, compute_lac: bool = True, compute_domain: bool = True):
        """Return global/local features and only the auxiliary heads requested by the run."""
        backbone_feat = self.extract_backbone_features(x)
        local_feat = self.local_projection(backbone_feat)
        global_emb = self.extract_global_embedding(backbone_feat)
        lac_logits = self.lac(local_feat) if compute_lac else None
        domain_logits = self.domain_classifier(self.grl(local_feat)) if compute_domain else None
        return global_emb, local_feat, lac_logits, domain_logits

    def freeze_lower_layers(self, num_blocks_to_freeze: int = 3, freeze_frontend: bool = True):
        """Đóng băng tầng frontend và các khối Conformer đầu tiên theo chiến lược Deep Freeze."""
        if freeze_frontend:
            if hasattr(self.feature_extractor, '_global_cmvn'):
                for p in self.feature_extractor._global_cmvn.parameters(): p.requires_grad = False
            if hasattr(self.feature_extractor, '_encoder') and hasattr(self.feature_extractor._encoder, 'embed'):
                for p in self.feature_extractor._encoder.embed.parameters(): p.requires_grad = False

        if hasattr(self.feature_extractor, '_encoder') and hasattr(self.feature_extractor._encoder, 'encoders'):
            for i, block in enumerate(self.feature_extractor._encoder.encoders):
                if i < num_blocks_to_freeze:
                    for p in block.parameters(): p.requires_grad = False
                else:
                    for p in block.parameters(): p.requires_grad = True

        if hasattr(self.feature_extractor, '_pool_layer'):
            for p in self.feature_extractor._pool_layer.parameters(): p.requires_grad = True
        for p in self.domain_classifier.parameters(): p.requires_grad = True
        for p in self.lac.parameters(): p.requires_grad = True

## Checkpoint, splits and gallery protocol


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

def load_compatible_checkpoint(path):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    cfg = payload.get('run_config', {})
    if cfg.get('pipeline_version') != 4 or cfg.get('embedding_dim') != 128:
        raise ValueError('Expected a pipeline-v4, 128-D checkpoint: ' + str(path))
    state = payload['model_state_dict']
    if 'lac.prototypes' not in state or state['lac.prototypes'].shape[-1] != 128:
        raise ValueError('Missing/incompatible LAC prototypes.')
    train_ids = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
    if payload.get('train_songs') != train_ids or len(train_ids) != state['lac.prototypes'].shape[0]:
        raise ValueError('Checkpoint train-song mapping does not match the dataset.')
    return payload

checkpoint_metadata = load_compatible_checkpoint(finetuned_ckpt_path)
EMBEDDING_DIM = 128
NUM_TRAIN_SONGS = checkpoint_metadata['model_state_dict']['lac.prototypes'].shape[0]
test_dataset = ByteHumCachedTripletDataset(str(TEST_DIR), prep, cqt_ext, is_training=False)
split_ids = {name: {p.name for p in directory.iterdir() if p.is_dir()}
             for name, directory in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]}
for left, right in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    if split_ids[left] & split_ids[right]:
        raise ValueError(f'Song leakage: {left}/{right}')

def resolve_gallery(dataset, mode=None):
    mode = GALLERY_MODE if mode is None else mode
    if mode == 'test':
        result = {}
        for song, fragments in dataset.song_data.items():
            result[song] = sorted({p for pool in fragments.values()
                                   for p in (pool['refs'] if pool['refs'] else pool['vocals'])})
    elif GALLERY_MANIFEST:
        manifest_path = Path(GALLERY_MANIFEST).resolve()
        mapping = json.loads(manifest_path.read_text(encoding='utf-8'))
        result = {}
        for song, paths in mapping.items():
            result[str(song)] = [str((manifest_path.parent / Path(p)).resolve())
                                for p in ([paths] if isinstance(paths, str) else paths)]
    else:
        if not MUSIC_DB_DIR.is_dir():
            raise FileNotFoundError(MUSIC_DB_DIR)
        result = {}
        for path in sorted(MUSIC_DB_DIR.iterdir()):
            if path.suffix.lower() in ('.wav', '.mp3', '.flac', '.ogg', '.m4a'):
                if path.stem in result:
                    raise ValueError('Duplicate gallery song ID; use GALLERY_MANIFEST: ' + path.stem)
                result[path.stem] = [str(path)]
    if not result or any(not paths for paths in result.values()):
        raise ValueError('Empty gallery or song without a reference.')
    for paths in result.values():
        for path in paths:
            if not Path(path).is_file():
                raise FileNotFoundError(path)
    missing = {song for song, _, _ in dataset.humming_samples} - result.keys()
    if missing:
        raise ValueError('Query ground truth missing from gallery: ' + str(sorted(missing)[:5]))
    query_paths = {str(Path(p).resolve()) for _, _, p in dataset.humming_samples}
    if query_paths & {str(Path(p).resolve()) for paths in result.values() for p in paths}:
        raise ValueError('Query/reference file overlap.')
    return dict(sorted(result.items()))

gallery_paths = resolve_gallery(test_dataset)
all_test_queries = list(test_dataset.humming_samples)
eval_queries = all_test_queries if MAX_EVAL_QUERIES is None else all_test_queries[:MAX_EVAL_QUERIES]
print('Best epoch:', checkpoint_metadata['epoch'], '| Val MRR:', checkpoint_metadata['best_val_mrr'])
print('Protocol:', GALLERY_MODE, '| Gallery:', len(gallery_paths), '| Queries:', len(eval_queries), '/', len(all_test_queries))


## Exact retrieval and FAISS candidate recall


In [ ]:
@torch.inference_mode()
def encode_cqts(eval_model, cqts):
    if not cqts:
        raise ValueError('No CQT chunks to encode.')
    gs, ls = [], []
    for offset in range(0, len(cqts), EVAL_BATCH_SIZE):
        batch = torch.stack(cqts[offset:offset + EVAL_BATCH_SIZE]).to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP_ENABLED):
            g, l, _, _ = eval_model(batch, compute_lac=False, compute_domain=False)
        gs.append(F.normalize(g.float(), dim=-1).cpu())
        ls.append(F.normalize(l.float(), dim=-1).cpu())
    g, l = torch.cat(gs), torch.cat(ls).reshape(-1, EMBEDDING_DIM)
    if not torch.isfinite(g).all() or not torch.isfinite(l).all():
        raise ValueError('Non-finite model features.')
    return g, l

def synchronize():
    if AMP_ENABLED:
        torch.cuda.synchronize()

def gallery_signature(paths, checkpoint_hash):
    records = [(song, str(Path(path).resolve()), Path(path).stat().st_size, Path(path).stat().st_mtime_ns)
               for song, files in paths.items() for path in files]
    return hashlib.sha256(json.dumps({'checkpoint': checkpoint_hash, 'records': records,
        'pipeline': 'v4_16k_10s_hop3_96x83', 'amp': AMP_ENABLED,
        'torch': torch.__version__}, sort_keys=True).encode()).hexdigest()

@torch.inference_mode()
def build_gallery(eval_model, paths, checkpoint_hash):
    signature = gallery_signature(paths, checkpoint_hash)
    cache_dir = STAGE2_ROOT / 'cache' / 'evaluation_features'
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / (signature + '.pt')
    if cache_path.is_file():
        cached = torch.load(cache_path, map_location='cpu', weights_only=True)
        if cached['signature'] != signature or list(cached['features']) != list(paths):
            raise ValueError('Gallery feature-cache mismatch: ' + str(cache_path))
        for g, l in cached['features'].values():
            if g.ndim != 2 or l.ndim != 2 or g.shape[1] != EMBEDDING_DIM or l.shape[1] != EMBEDDING_DIM or not len(g) or not len(l) or not torch.isfinite(g).all() or not torch.isfinite(l).all():
                raise ValueError('Invalid gallery feature-cache: ' + str(cache_path))
        return cached['features'], signature
    features = {}
    for song, files in tqdm(paths.items(), desc='Encode gallery'):
        parts = [encode_cqts(eval_model, test_dataset._get_raw_cqts(path)) for path in files]
        features[song] = (torch.cat([g for g, _ in parts]), torch.cat([l for _, l in parts]))
    temp_path = cache_path.with_suffix('.tmp')
    torch.save({'signature': signature, 'features': features}, temp_path)
    os.replace(temp_path, cache_path)
    return features, signature

@torch.inference_mode()
def score_gallery(query_global, query_local, gallery):
    # Bound padding to one song group, and similarities to reference-frame blocks.
    qg, ql = query_global.to(device), query_local.to(device)
    songs = list(gallery)
    local_scores, global_scores = [], []
    for offset in range(0, len(songs), GALLERY_SONG_BLOCK):
        group = [gallery[s] for s in songs[offset:offset + GALLERY_SONG_BLOCK]]
        gp = nn.utils.rnn.pad_sequence([g for g, _ in group], batch_first=True).to(device)
        gm = torch.arange(gp.shape[1], device=device)[None, :] < torch.tensor([len(g) for g, _ in group], device=device)[:, None]
        similarities = torch.einsum('qd,sgd->sqg', qg, gp).masked_fill(~gm[:, None, :], float('-inf'))
        global_scores.append(similarities.amax(dim=(1, 2)).cpu())
        max_per_query = torch.full((len(group), len(ql)), float('-inf'), device=device)
        longest = max(len(l) for _, l in group)
        for frame_start in range(0, longest, REFERENCE_FRAME_BLOCK):
            slices = [l[frame_start:frame_start + REFERENCE_FRAME_BLOCK] for _, l in group]
            padded = nn.utils.rnn.pad_sequence(slices, batch_first=True).to(device)
            mask = torch.arange(padded.shape[1], device=device)[None, :] < torch.tensor([len(l) for l in slices], device=device)[:, None]
            sims = torch.einsum('qd,sgd->sqg', ql, padded).masked_fill(~mask[:, None, :], float('-inf'))
            max_per_query = torch.maximum(max_per_query, sims.amax(dim=-1))
        local_scores.append(max_per_query.mean(dim=-1).cpu())
    local, global_ = torch.cat(local_scores), torch.cat(global_scores)
    return {'local': local, 'global': global_, 'fused': 0.5 * (local + global_)}

def tie_statistics(scores, target):
    values = np.asarray(scores, dtype=np.float64)
    if not np.isfinite(values[target]):
        return float('inf'), 0, float('inf')
    tied = np.isclose(values, values[target], rtol=1e-6, atol=1e-7)
    # Treat close scores as one tie group, excluding them from greater counts.
    greater = int(((values > values[target]) & ~tied).sum())
    ties = int(tied.sum())
    return 1.0 + greater + 0.5 * (ties - 1), ties, greater

def summarize_rows(frame, mode):
    ranks = frame[mode + '_rank'].to_numpy()
    ties = frame[mode + '_ties'].to_numpy()
    greater = frame[mode + '_greater'].to_numpy()
    valid = np.isfinite(ranks)
    result = {'mrr': float(np.mean(np.where(valid, 1.0 / ranks, 0.0))),
              'mean_rank': float(ranks.mean()) if valid.all() else None,
              'retrieved_queries': int(valid.sum()), 'queries': len(frame)}
    cmc = []
    for k in range(1, 21):
        expected_hits = np.zeros(len(frame))
        expected_hits[valid] = np.clip((k - greater[valid]) / ties[valid], 0, 1)
        cmc.append(float(expected_hits.mean()))
    result.update({f'recall_at_{k}': cmc[k - 1] for k in (1, 5, 10, 20)})
    result['cmc'] = cmc
    return result

def build_faiss_gallery(gallery):
    if faiss is None:
        return None, None
    songs = list(gallery)
    matrix = np.ascontiguousarray(torch.cat([gallery[s][0] for s in songs]).numpy(), dtype=np.float32)
    faiss.normalize_L2(matrix)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(matrix)
    ids = np.concatenate([np.full(len(gallery[s][0]), i, dtype=np.int64) for i, s in enumerate(songs)])
    return index, ids

def candidate_ids(query_global, index, chunk_to_song, k):
    if k < 1:
        raise ValueError('FAISS k must be positive.')
    query = np.ascontiguousarray(query_global.numpy(), dtype=np.float32)
    _, nearest = index.search(query, min(k, index.ntotal))
    return set(int(chunk_to_song[i]) for i in nearest.ravel() if i >= 0)

@torch.inference_mode()
def evaluate_model(eval_model, gallery, queries, label):
    if not queries:
        raise ValueError('No queries; no result can be reported.')
    songs = list(gallery)
    song_to_index = {s: i for i, s in enumerate(songs)}
    index, chunk_map = build_faiss_gallery(gallery) if RUN_FAISS_BENCHMARK else (None, None)
    # Warm model/CUDA before timing; reported CQT/I/O remains cache-state-dependent.
    encode_cqts(eval_model, test_dataset._get_raw_cqts(queries[0][2])[:1])
    synchronize()
    rows, query_scores = [], {}
    for song, fragment, path in tqdm(queries, desc='Queries ' + label):
        if song not in song_to_index:
            raise ValueError('Missing ground truth: ' + song)
        start = time.perf_counter()
        cqts = test_dataset._get_raw_cqts(path)
        cqt_ms = (time.perf_counter() - start) * 1000
        synchronize()
        start_model = time.perf_counter()
        qg, ql = encode_cqts(eval_model, cqts)
        synchronize()
        model_ms = (time.perf_counter() - start_model) * 1000
        start_score = time.perf_counter()
        scores = score_gallery(qg, ql, gallery)
        synchronize()
        score_ms = (time.perf_counter() - start_score) * 1000
        row = {'song': song, 'fragment': fragment, 'query': path, 'cqt_io_ms': cqt_ms,
               'model_ms': model_ms, 'exact_scoring_ms': score_ms}
        if index is not None:
            gate_start = time.perf_counter()
            candidates = candidate_ids(qg, index, chunk_map, FAISS_CHUNK_K)
            row['faiss_ms'] = (time.perf_counter() - gate_start) * 1000
            row['faiss_candidate_hit'] = song_to_index[song] in candidates
            row['faiss_candidate_songs'] = len(candidates)
            gated = torch.full_like(scores['fused'], float('-inf'))
            kept = sorted(candidates)
            gated[kept] = scores['fused'][kept]
            scores['faiss_fused'] = gated
        for mode, values in scores.items():
            rank, ties, greater = tie_statistics(values.numpy(), song_to_index[song])
            row.update({mode + '_rank': rank, mode + '_ties': ties, mode + '_greater': greater,
                        mode + '_top1_song': songs[int(values.argmax())]})
        rows.append(row)
        query_scores[path] = scores['fused'].clone()
    frame = pd.DataFrame(rows)
    modes = ['local', 'global', 'fused'] + (['faiss_fused'] if index is not None else [])
    result = {'label': label, 'queries': len(frame), 'gallery_songs': len(songs),
              'modes': {mode: summarize_rows(frame, mode) for mode in modes},
              'latency': {column: {'p50_ms': float(frame[column].median()),
                                   'p95_ms': float(frame[column].quantile(.95))}
                          for column in ('cqt_io_ms', 'model_ms', 'exact_scoring_ms')}}
    if index is not None:
        result['faiss'] = {'kind': 'flat', 'chunk_k_per_query_chunk': FAISS_CHUNK_K,
                          'candidate_recall': float(frame.faiss_candidate_hit.mean()),
                          'mean_candidate_songs': float(frame.faiss_candidate_songs.mean())}
    frame.to_csv(OUTPUT_DIR / (label + '_queries.csv'), index=False)
    return result, frame, query_scores


In [ ]:
# Keep validation for iteration; use test only for final reporting.
QUERY_SPLIT = os.environ.get('QBH_QUERY_SPLIT', 'val')  # 'val' or 'test'
if QUERY_SPLIT == 'val':
    query_dataset = ByteHumCachedTripletDataset(str(VAL_DIR), prep, cqt_ext, is_training=False)
elif QUERY_SPLIT == 'test':
    query_dataset = test_dataset
else:
    raise ValueError("QUERY_SPLIT must be 'val' or 'test'.")
eval_queries = list(query_dataset.humming_samples)
all_test_queries = eval_queries
if not eval_queries:
    raise ValueError(f'No {QUERY_SPLIT} queries found.')
OUTPUT_DIR = STAGE2_ROOT / 'runs' / 'local_4gb' / f'retrieval_ablation_full_{QUERY_SPLIT}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Query split: {QUERY_SPLIT} | Queries: {len(eval_queries)} | Gallery songs: {len(gallery_paths)}')


## Candidate recall: global vs. local vs. union

The experiment indexes all cached local frame features on CPU. Query embeddings are cached on disk; only nearest-neighbor candidate recall is measured here. Final reranking quality is evaluated in the next step using the candidate limit selected on validation.

In [ ]:
from collections import defaultdict

if GALLERY_MODE != 'full' or len(gallery_paths) != 300 or not eval_queries:
    raise ValueError('This ablation requires the complete 300-song gallery and a non-empty query split.')
if faiss is None:
    raise ImportError('Install faiss-cpu to run the candidate-recall ablation.')
faiss.omp_set_num_threads(max(1, min(4, os.cpu_count() or 1)))

local_hash = file_sha256(finetuned_ckpt_path)
model = ConformerByteHumModel(num_classes=NUM_TRAIN_SONGS,
                              embedding_dim=EMBEDDING_DIM,
                              allow_random_init=True)
model.load_state_dict(checkpoint_metadata['model_state_dict'], strict=True)
model = model.to(device).eval()
gallery, gallery_signature_value = build_gallery(model, gallery_paths, local_hash)

# Cache query embeddings separately from gallery features; input order and file
# fingerprints are part of the key so a changed query set cannot reuse stale data.
query_records = [(song, fragment, str(Path(path).resolve()),
                  Path(path).stat().st_size, Path(path).stat().st_mtime_ns)
                 for song, fragment, path in eval_queries]
query_signature = hashlib.sha256(json.dumps({
    'checkpoint': local_hash,
    'gallery': gallery_signature_value,
    'queries': query_records,
    'pipeline': 'v4_16k_10s_hop3_96x83',
    'torch': torch.__version__
}, sort_keys=True).encode()).hexdigest()
query_cache_path = STAGE2_ROOT / 'cache' / 'evaluation_features' / ('queries_' + query_signature + '.pt')
if query_cache_path.is_file():
    payload = torch.load(query_cache_path, map_location='cpu', weights_only=True)
    if payload.get('signature') != query_signature or len(payload.get('features', [])) != len(eval_queries):
        raise ValueError('Query feature cache does not match this checkpoint/query set.')
    query_features = payload['features']
    print('Reused cached query embeddings:', query_cache_path)
else:
    query_features = []
    for song, fragment, path in tqdm(eval_queries, desc='Encode queries'):
        cqts = query_dataset._get_raw_cqts(path)
        query_features.append(encode_cqts(model, cqts))
    temp_path = query_cache_path.with_suffix('.tmp')
    torch.save({'signature': query_signature, 'features': query_features}, temp_path)
    os.replace(temp_path, query_cache_path)
    print('Saved query embeddings:', query_cache_path)

song_ids = list(gallery)
song_to_index = {song: i for i, song in enumerate(song_ids)}
global_index, chunk_to_song = build_faiss_gallery(gallery)

# Pool local frames within each chunk for a compact complementary candidate
# index; reranking can still use the unpooled local sequence in a later step.
LOCAL_FRAMES_PER_CHUNK = 20
local_parts = [gallery[song][1].reshape(-1, LOCAL_FRAMES_PER_CHUNK, EMBEDDING_DIM)
               .mean(dim=1)
               for song in song_ids]
local_matrix = np.ascontiguousarray(torch.cat(local_parts).numpy(), dtype=np.float32)
local_to_song = np.concatenate([
    np.full(len(local_parts[i]), i, dtype=np.int64)
    for i, song in enumerate(song_ids)
])
faiss.normalize_L2(local_matrix)
local_index = faiss.IndexFlatIP(EMBEDDING_DIM)
local_index.add(local_matrix)
del local_matrix, local_parts
print('Global/local index vectors:', global_index.ntotal, '/', local_index.ntotal,
      '| local features are mean-pooled within each audio chunk.')

def batched_faiss_search(index, queries, k, batch_size=128):
    """Bound the exact-index distance matrix to one small query batch."""
    batches = [index.search(queries[start:start + batch_size], k)[1]
               for start in range(0, len(queries), batch_size)]
    return np.concatenate(batches, axis=0)

global_rows, global_slices = [], []
local_rows, local_slices = [], []
cursor = 0
for global_features, local_features in query_features:
    g = np.ascontiguousarray(global_features.numpy(), dtype=np.float32)
    l = np.ascontiguousarray(local_features.reshape(
        -1, LOCAL_FRAMES_PER_CHUNK, EMBEDDING_DIM).mean(dim=1).numpy(), dtype=np.float32)
    global_rows.append(g)
    global_slices.append(slice(cursor, cursor + len(g)))
    cursor += len(g)
    local_rows.append(l)
    local_slices.append(slice(sum(len(x) for x in local_rows[:-1]),
                              sum(len(x) for x in local_rows)))
global_matrix = np.ascontiguousarray(np.concatenate(global_rows), dtype=np.float32)
local_query_matrix = np.ascontiguousarray(np.concatenate(local_rows), dtype=np.float32)
faiss.normalize_L2(global_matrix)
faiss.normalize_L2(local_query_matrix)

K_VALUES = (10, 25, 50, 100, 200, 300, 500, 800, 1200)
MAX_K = max(K_VALUES)
global_neighbors = batched_faiss_search(global_index, global_matrix, MAX_K)
print('Global neighbor search complete.')
local_neighbors = batched_faiss_search(local_index, local_query_matrix, MAX_K)
print('Local neighbor search complete.')
rows = []
per_query = []
for query_i, ((song, fragment, path), g_slice, l_slice) in enumerate(
        zip(eval_queries, global_slices, local_slices)):
    target = song_to_index[song]
    g_hits = global_neighbors[g_slice]
    l_hits = local_neighbors[l_slice]
    for k in K_VALUES:
        global_candidates = set(int(chunk_to_song[j]) for j in g_hits[:, :k].ravel() if j >= 0)
        local_candidates = set(int(local_to_song[j]) for j in l_hits[:, :k].ravel() if j >= 0)
        strategies = {
            'global_only': global_candidates,
            'local_only': local_candidates,
            'global_plus_local': global_candidates | local_candidates,
        }
        for strategy, candidates in strategies.items():
            hit = target in candidates
            row = {'query_index': query_i, 'song': song, 'fragment': fragment,
                   'query': path, 'neighbors_per_query_vector': k,
                   'strategy': strategy, 'candidate_songs': len(candidates),
                   'ground_truth_hit': hit}
            rows.append(row)
            per_query.append(row)

frame = pd.DataFrame(rows)
summary = (frame.groupby(['neighbors_per_query_vector', 'strategy'], sort=False)
           .agg(queries=('ground_truth_hit', 'size'),
                candidate_recall=('ground_truth_hit', 'mean'),
                mean_candidate_songs=('candidate_songs', 'mean'),
                p95_candidate_songs=('candidate_songs', lambda x: float(x.quantile(.95))))
           .reset_index())
summary['candidate_recall_pct'] = 100 * summary['candidate_recall']
summary.to_csv(OUTPUT_DIR / 'candidate_recall_summary.csv', index=False)
frame.to_csv(OUTPUT_DIR / 'candidate_recall_queries.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 5.5))
for strategy, group in summary.groupby('strategy', sort=False):
    ax.plot(group['mean_candidate_songs'], group['candidate_recall_pct'], marker='o', label=strategy)
ax.axhline(95, color='gray', linestyle='--', linewidth=1, label='95% target')
ax.set(xlabel='Mean unique candidate songs', ylabel='Ground-truth candidate recall (%)',
       title='Candidate recall vs. rerank workload (300-song gallery)')
ax.grid(True, alpha=.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'candidate_recall_curve.png', dpi=180)
plt.show()

metadata = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'protocol': {'gallery_mode': GALLERY_MODE, 'query_split': QUERY_SPLIT,
                 'gallery_songs': len(song_ids), 'queries': len(eval_queries), 'subset': False,
                 'neighbors_per_query_vector': list(K_VALUES),
                 'strategies': ['global_only', 'local_only', 'global_plus_local']},
    'checkpoint': str(finetuned_ckpt_path), 'checkpoint_sha256': local_hash,
    'gallery_signature': gallery_signature_value, 'query_signature': query_signature,
    'local_index_vectors': int(len(local_to_song)),
    'method': 'FAISS IndexFlatIP over global chunks and mean-pooled local chunk embeddings; no model training.',
    'interpretation': 'Candidate recall measures whether the ground-truth song survives the gate; it is not final reranked Recall@K.'
}
(OUTPUT_DIR / 'candidate_recall_metadata.json').write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
display(summary[['neighbors_per_query_vector', 'strategy', 'candidate_recall_pct',
                 'mean_candidate_songs', 'p95_candidate_songs']])
print('Saved candidate-recall ablation:', OUTPUT_DIR)


## Temporal-windowed local scoring

This exact 300-song comparison constrains each query chunk to align inside a contiguous reference window at a few duration ratios. It reports local-only and fused ranks alongside the existing exact fused baseline. Use validation to choose window scales; reserve test for one final report.

In [ ]:
@torch.inference_mode()
def score_windowed_local(query_local, gallery_features, window_scales=(0.75, 1.0, 1.25, 1.5)):
    """Align query chunks to one continuous reference window before MaxMean scoring."""
    q = query_local.to(device).reshape(-1, LOCAL_FRAMES_PER_CHUNK, EMBEDDING_DIM)
    query_chunks = q.shape[0]
    song_ids = list(gallery_features)
    output = []
    q_index = torch.arange(query_chunks, device=device)[:, None]
    for offset in range(0, len(song_ids), GALLERY_SONG_BLOCK):
        group_songs = song_ids[offset:offset + GALLERY_SONG_BLOCK]
        group = [gallery_features[song][1].reshape(-1, LOCAL_FRAMES_PER_CHUNK, EMBEDDING_DIM)
                 for song in group_songs]
        lengths = torch.tensor([len(x) for x in group], device=device)
        reference = nn.utils.rnn.pad_sequence(group, batch_first=True).to(device)
        chunk_mask = torch.arange(reference.shape[1], device=device)[None, :] < lengths[:, None]
        # [song, query_chunk, reference_chunk, query_frame, reference_frame]
        similarity = torch.einsum('atd,grud->gartu', q, reference)
        similarity = similarity.masked_fill(~chunk_mask[:, None, :, None, None], float('-inf'))
        pair_scores = similarity.amax(dim=-1).mean(dim=-1)  # [song, query_chunk, reference_chunk]
        for song_i, ref_length in enumerate(lengths.tolist()):
            best = torch.tensor(float('-inf'), device=device)
            for scale in window_scales:
                window_chunks = max(1, int(round(query_chunks * scale)))
                if window_chunks > ref_length:
                    continue
                starts = torch.arange(ref_length - window_chunks + 1, device=device)
                if query_chunks == 1:
                    positions = torch.zeros(1, dtype=torch.long, device=device)
                else:
                    positions = torch.round(torch.arange(query_chunks, device=device)
                        * (window_chunks - 1) / (query_chunks - 1)).long()
                reference_positions = starts[None, :] + positions[:, None]
                aligned = pair_scores[song_i, q_index, reference_positions]
                best = torch.maximum(best, aligned.mean(dim=0).amax())
            output.append(best)
    return torch.stack(output).cpu()


rows = []
window_scores = []
for query_i, ((song, fragment, path), (query_global, query_local)) in enumerate(
        zip(eval_queries, query_features)):
    baseline = score_gallery(query_global, query_local, gallery)
    local_window = score_windowed_local(query_local, gallery)
    fused_window = 0.5 * (local_window + baseline['global'])
    target = song_to_index[song]
    row = {'query_index': query_i, 'query': path, 'song': song, 'fragment': fragment}
    for name, value in [('baseline_fused', baseline['fused']),
                        ('windowed_local', local_window),
                        ('windowed_fused', fused_window)]:
        rank, ties, greater = tie_statistics(value.numpy(), target)
        row.update({name + '_rank': rank, name + '_ties': ties, name + '_greater': greater,
                    name + '_top1_song': song_ids[int(value.argmax())]})
    rows.append(row)
    window_scores.append({'baseline_global': baseline['global'],
                          'baseline_fused': baseline['fused'],
                          'windowed_local': local_window,
                          'windowed_fused': fused_window})
    if (query_i + 1) % 50 == 0:
        print(f'Windowed exact scoring: {query_i + 1}/{len(eval_queries)} queries', flush=True)

window_frame = pd.DataFrame(rows)
window_metrics = {'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'protocol': {'gallery_mode': GALLERY_MODE, 'query_split': QUERY_SPLIT,
                 'gallery_songs': len(song_ids),
                 'queries': len(window_frame), 'subset': len(window_frame) != len(all_test_queries),
                 'window_scales': [0.75, 1.0, 1.25, 1.5],
                 'local_scoring': 'query/reference chunk MaxMean aligned within one contiguous reference window',
                 'fusion': '0.5 windowed local + 0.5 existing global maximum cosine'},
    'checkpoint_sha256': local_hash, 'gallery_signature': gallery_signature_value,
    'results': {mode: summarize_rows(window_frame, mode)
                for mode in ('baseline_fused', 'windowed_local', 'windowed_fused')}}
window_frame.to_csv(OUTPUT_DIR / 'windowed_scoring_queries.csv', index=False)
(OUTPUT_DIR / 'windowed_scoring_metrics.json').write_text(
    json.dumps(window_metrics, indent=2, allow_nan=False), encoding='utf-8')
display(pd.DataFrame([{'mode': mode, 'MRR': result['mrr'],
                       'Top-1': result['recall_at_1'], 'Top-5': result['recall_at_5'],
                       'Top-10': result['recall_at_10'], 'Top-20': result['recall_at_20']}
                      for mode, result in window_metrics['results'].items()]))
print('Saved windowed-scoring comparison:', OUTPUT_DIR)


## Candidate-limited reranking at K=500

The global/local union is exact-reranked on a subset of the full gallery and compared per query with the saved full-gallery result. Paired bootstrap intervals show both ranking impact and candidate recall.

In [ ]:
RERANK_K = 500
baseline_path = OUTPUT_DIR / 'windowed_scoring_queries.csv'
if QUERY_SPLIT == 'test' and not baseline_path.is_file():
    baseline_path = STAGE2_ROOT / 'runs' / 'local_4gb' / 'retrieval_ablation_full' / 'windowed_scoring_queries.csv'
if not baseline_path.is_file():
    raise FileNotFoundError('Run the temporal-scoring cell first: ' + str(baseline_path))
baseline_frame = pd.read_csv(baseline_path)
if len(baseline_frame) != len(eval_queries) or baseline_frame['query_index'].tolist() != list(range(len(eval_queries))):
    raise ValueError('Full-gallery baseline does not match the current query split/order.')

rerank_rows = []
for query_i, ((song, fragment, path), (query_global, query_local), g_slice, l_slice) in enumerate(
        zip(eval_queries, query_features, global_slices, local_slices)):
    global_candidates = {int(chunk_to_song[j]) for j in global_neighbors[g_slice, :RERANK_K].ravel() if j >= 0}
    local_candidates = {int(local_to_song[j]) for j in local_neighbors[l_slice, :RERANK_K].ravel() if j >= 0}
    candidate_indices = sorted(global_candidates | local_candidates)
    candidate_songs = [song_ids[i] for i in candidate_indices]
    candidate_gallery = {candidate_song: gallery[candidate_song] for candidate_song in candidate_songs}
    scores = score_gallery(query_global, query_local, candidate_gallery)['fused'].numpy()
    row = {'query_index': query_i, 'query': path, 'song': song,
           'candidate_count': len(candidate_songs), 'candidate_hit': song in candidate_gallery}
    if song in candidate_gallery:
        target = candidate_songs.index(song)
        rank, ties, greater = tie_statistics(scores, target)
    else:
        rank, ties, greater = float('inf'), 0, float('inf')
    row.update({'candidate_fused_rank': rank, 'candidate_fused_ties': ties,
                'candidate_fused_greater': greater,
                'baseline_fused_rank': baseline_frame.loc[query_i, 'baseline_fused_rank'],
                'baseline_fused_ties': baseline_frame.loc[query_i, 'baseline_fused_ties'],
                'baseline_fused_greater': baseline_frame.loc[query_i, 'baseline_fused_greater']})
    rerank_rows.append(row)
    if (query_i + 1) % 100 == 0:
        print(f'Candidate reranking: {query_i + 1}/{len(eval_queries)} queries', flush=True)

rerank_frame = pd.DataFrame(rerank_rows)
rerank_metrics = {'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'protocol': {'query_split': QUERY_SPLIT, 'gallery_mode': GALLERY_MODE,
                 'gallery_songs': len(song_ids), 'queries': len(eval_queries),
                 'candidate_strategy': 'global_plus_local', 'neighbors_per_query_vector': RERANK_K,
                 'reranker': 'exact existing MaxMean local + max-cosine global, fused 0.5/0.5'},
    'candidate_recall': float(rerank_frame['candidate_hit'].mean()),
    'mean_candidate_songs': float(rerank_frame['candidate_count'].mean()),
    'results': {'full_gallery': summarize_rows(baseline_frame, 'baseline_fused'),
                'candidate_reranked': summarize_rows(rerank_frame, 'candidate_fused')}}
comparison = []
for name, column, k in [('MRR', None, 0), ('Top-1', None, 1), ('Top-5', None, 5),
                        ('Top-10', None, 10), ('Top-20', None, 20)]:
    def expected_metric(frame, prefix, cutoff):
        rank = frame[prefix + '_rank'].to_numpy()
        if cutoff == 0:
            return np.where(np.isfinite(rank), 1.0 / rank, 0.0)
        ties = frame[prefix + '_ties'].to_numpy()
        greater = frame[prefix + '_greater'].to_numpy()
        valid = np.isfinite(rank)
        values = np.zeros(len(frame))
        values[valid] = np.clip((cutoff - greater[valid]) / ties[valid], 0, 1)
        return values
    base_values = expected_metric(rerank_frame, 'baseline_fused', k)
    candidate_values = expected_metric(rerank_frame, 'candidate_fused', k)
    delta = candidate_values - base_values
    rng = np.random.default_rng(20261003 + k)
    bootstrap = np.array([delta[rng.integers(0, len(delta), len(delta))].mean()
                          for _ in range(5000)])
    comparison.append({'metric': name, 'full_gallery': float(base_values.mean()),
        'candidate_reranked': float(candidate_values.mean()), 'delta': float(delta.mean()),
        'delta_ci95_low': float(np.quantile(bootstrap, 0.025)),
        'delta_ci95_high': float(np.quantile(bootstrap, 0.975))})
rerank_metrics['paired_comparison'] = comparison
rerank_frame.to_csv(OUTPUT_DIR / 'candidate_rerank_queries.csv', index=False)
(OUTPUT_DIR / 'candidate_rerank_metrics.json').write_text(
    json.dumps(rerank_metrics, indent=2, allow_nan=False), encoding='utf-8')
display(pd.DataFrame(comparison))
print('Saved candidate-limited reranking comparison:', OUTPUT_DIR)
